# 🛡️ HỆ THỐNG PHÁT HIỆN XÂM NHẬP MẠNG (NIDS) BẰNG MACHINE LEARNING & DEEP LEARNING

**Dataset:** UNSW-NB15 · CIC-IDS2017 &nbsp;|&nbsp; **Mô hình:** Random Forest (baseline) · 1D-CNN + BiLSTM &nbsp;|&nbsp; **Bài toán:** (A) nhị phân Normal/Attack · (B) đa lớp loại tấn công

> Notebook dùng **một pipeline chung** gồm các hàm tái sử dụng `load → clean → preprocess → train → evaluate` cho cả 2 dataset. Nhờ vậy không lặp code và có thể so sánh trực tiếp kết quả giữa hai dataset ở mục 8.

**Mục lục**
0. Giới thiệu
1. Cài đặt & import thư viện, cấu hình SEED, kiểm tra GPU
2. Load dữ liệu
3. EDA – Phân tích khám phá dữ liệu
4. Tiền xử lý dữ liệu
5. Mô hình 1 – Baseline Random Forest
6. Mô hình 2 – Deep Learning 1D-CNN + BiLSTM
7. Đánh giá kết quả trên tập test
8. So sánh hai mô hình
9. Demo dự đoán
10. Lưu kết quả
11. Kết luận & hướng phát triển
12. Tài liệu tham khảo

## 0. Giới thiệu

### 0.1 Bối cảnh
Hệ thống phát hiện xâm nhập mạng (**Network Intrusion Detection System – NIDS**) giám sát lưu lượng mạng (gói tin hoặc luồng – *flow*) để phát hiện hành vi độc hại như DoS/DDoS, quét cổng, brute-force, khai thác lỗ hổng, botnet… NIDS là một lớp phòng thủ quan trọng bên cạnh firewall: firewall chặn theo luật truy cập, còn NIDS **phân tích nội dung/hành vi** để cảnh báo tấn công đã lọt qua.

### 0.2 Signature-based và Anomaly-based
| Tiêu chí | Signature-based (dựa trên chữ ký) | Anomaly-based / ML-based (dựa trên hành vi) |
|---|---|---|
| Nguyên lý | So khớp mẫu (chữ ký) của tấn công đã biết, vd Snort, Suricata | Học từ dữ liệu hành vi bình thường và/hoặc mẫu tấn công, cảnh báo khi có sai lệch |
| Ưu điểm | Rất chính xác với tấn công đã biết, ít báo động giả, dễ giải thích | Có thể phát hiện biến thể mới / zero-day, tự học từ dữ liệu |
| Nhược điểm | Không phát hiện được tấn công mới, phải cập nhật luật liên tục | Tỉ lệ báo động giả (FAR) cao hơn, phụ thuộc chất lượng dữ liệu huấn luyện, khó giải thích |

Project này đi theo hướng **ML-based (học có giám sát)** trên các **đặc trưng luồng (flow features)** trích xuất từ lưu lượng mạng.

### 0.3 Mục tiêu
1. Xây dựng pipeline tiền xử lý **chống rò rỉ dữ liệu**, dùng lại được cho nhiều dataset.
2. Huấn luyện và so sánh **Random Forest** (ML truyền thống) với **1D-CNN + BiLSTM** (Deep Learning) cho cả 2 bài toán nhị phân và đa lớp.
3. Đánh giá bằng các chỉ số phù hợp với IDS trên dữ liệu mất cân bằng: **F1-macro, Detection Rate, False Alarm Rate, ROC-AUC, PR-AUC**, kèm thời gian train/dự đoán và kích thước mô hình.
4. Phân tích lỗi, giải thích mô hình (feature importance, SHAP) và bàn về khả năng triển khai thời gian thực.

### 0.4 Mô tả dataset
| | **UNSW-NB15** | **CIC-IDS2017** |
|---|---|---|
| Nguồn | Cyber Range Lab – UNSW Canberra (Moustafa & Slay, 2015), sinh bằng IXIA PerfectStorm | Canadian Institute for Cybersecurity – UNB (Sharafaldin et al., 2018) |
| Số mẫu | Bản train/test chính thức: 175.341 / 82.332 bản ghi | ~2,83 triệu luồng, 8 file CSV ghi trong 5 ngày (thứ 2 → thứ 6) |
| Đặc trưng | 42 đặc trưng (Argus, Bro-IDS) + `attack_cat`, `label` | ~78 đặc trưng trích xuất bằng CICFlowMeter + `Label` |
| Lớp | Normal + 9 loại: Fuzzers, Analysis, Backdoor, DoS, Exploits, Generic, Reconnaissance, Shellcode, Worms | BENIGN + 14 loại: DoS (Hulk, GoldenEye, slowloris, Slowhttptest), DDoS, PortScan, FTP-/SSH-Patator, Web Attack (Brute Force, XSS, SQL Injection), Bot, Infiltration, Heartbleed |
| Chia tập | Có sẵn train/test → dùng đúng split gốc, tách validation từ train | Không có → chia phân tầng train/val/test = 70/15/15 sau khi loại trùng lặp |

### 0.5 Hướng dẫn chạy trên Kaggle
1. **Add Input**: thêm một bản UNSW-NB15 (có `UNSW_NB15_training-set` và `UNSW_NB15_testing-set`, dạng csv hoặc parquet) và một bản CIC-IDS2017 (8 file CSV theo ngày, vd thư mục *MachineLearningCVE*). Chỉ nên thêm **một** bản cho mỗi dataset để tránh trùng dữ liệu.
2. **Settings → Accelerator**: GPU T4 hoặc P100.
3. **Run All**. Có thể chỉnh `CONFIG` ở mục 1.2: chỉ chạy 1 dataset, đổi tỉ lệ lấy mẫu CIC-IDS2017, tắt tuning/SHAP để chạy nhanh hơn.
4. Thời gian chạy ước tính với cấu hình mặc định: khoảng 1–2 giờ, phần lớn dành cho CIC-IDS2017.

## 1. Cài đặt & Import thư viện, cấu hình SEED, kiểm tra GPU

**Làm gì:** import thư viện, cố định `SEED = 42` cho `random`, `numpy`, `tensorflow`, bật *memory growth* cho GPU.

**Tại sao:** cố định seed giúp tái lập kết quả. Lưu ý: một số phép toán cuDNN trên GPU vẫn có thể không tất định tuyệt đối, nên kết quả DL có thể lệch rất nhỏ giữa các lần chạy. Memory growth tránh việc TensorFlow chiếm toàn bộ VRAM ngay từ đầu.

Mọi thư viện dùng ở đây đều có sẵn trên Kaggle: pandas, numpy, matplotlib, seaborn, scikit-learn, tensorflow/keras, shap.

In [ ]:
# ===== 1.1 Import thư viện =====
import os, gc, re, copy, time, json, random, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.stats import chi2
from IPython.display import display

import sklearn
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_curve, auc,
                             roc_auc_score, precision_recall_curve, average_precision_score)

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Bật memory growth cho GPU (phải làm trước khi GPU được khởi tạo)
GPUS = tf.config.list_physical_devices('GPU')
for g in GPUS:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception as e:
        print('Không bật được memory growth:', e)

SEED = 42
def set_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed()

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 100

print('pandas', pd.__version__, '| numpy', np.__version__, '| scikit-learn', sklearn.__version__)
print('TensorFlow', tf.__version__, '| Keras', getattr(keras, '__version__', '?'))
print('GPU:', GPUS if GPUS else 'KHÔNG có GPU – phần Deep Learning sẽ chạy trên CPU (chậm hơn nhiều)')

### 1.2 Cấu hình chung & hàm tiện ích

Mọi tham số quan trọng được gom vào `CONFIG` để dễ thay đổi:
- `DATASETS`: danh sách dataset cần chạy (mặc định cả hai; xóa bớt một tên nếu chỉ muốn chạy một dataset, mọi cell đều tự bỏ qua dataset không có trong danh sách).
- `UNSW_PREFERRED_DIR`: UNSW-NB15 lấy từ `/kaggle/input/notebooks/khairulislam/unsw-nb15-eda` (`train.csv`, `test.csv`; `features.csv` chỉ là bảng mô tả đặc trưng nên tự bị bỏ qua). Nếu file có cột index thừa (`Unnamed: 0`), notebook tự loại để tránh rò rỉ.
- `CIC_PREFERRED_DIR`: input Kaggle có 2 bản CSV của CIC-IDS2017 cùng nội dung: `TrafficLabelling` (GeneratedLabelledFlows, có thêm Flow ID/IP/Timestamp, nhiều dòng rỗng) và `MachineLearningCVE`. Notebook chọn **MachineLearningCVE** vì sạch hơn và không chứa cột định danh. Nạp cả hai sẽ làm trùng dữ liệu. Thư mục `pcaps-dataset` (pcap thô, không phải CSV) tự động bị bỏ qua.
- `CIC_SAMPLE_FRAC`: CIC-IDS2017 có ~2,5 triệu dòng sau khi loại trùng. Để vừa thời gian/RAM của Kaggle, mặc định **lấy mẫu phân tầng 50%**. Lớp hiếm được giữ tối thiểu `CIC_MIN_PER_CLASS` mẫu (hoặc toàn bộ nếu ít hơn), nên **không mất lớp hiếm**. Đặt `1.0` để dùng toàn bộ dữ liệu.
- `RF_TUNE`, `RUN_SHAP`: tắt đi nếu muốn chạy nhanh.

`DS_INFO` mô tả riêng từng dataset: tên lớp bình thường, cột định danh/rò rỉ cần bỏ, cột phân loại, các đặc trưng quan trọng dùng trong EDA.

In [ ]:
# ===== 1.2 Cấu hình chung =====
CONFIG = {
    'DATASETS': ['UNSW-NB15', 'CIC-IDS2017'],  # bỏ bớt 1 tên nếu chỉ muốn chạy 1 dataset
    'UNSW_PREFERRED_DIR': 'unsw-nb15-eda',      # ưu tiên train.csv/test.csv trong thư mục này
    'CIC_PREFERRED_DIR': 'MachineLearningCVE',  # ưu tiên bản CSV này (không có Flow ID/IP/Timestamp)
    'CIC_SAMPLE_FRAC': 0.5,       # lấy mẫu phân tầng CIC-IDS2017 sau khi loại trùng (1.0 = toàn bộ)
    'CIC_MIN_PER_CLASS': 5000,    # lớp hiếm giữ tối thiểu số mẫu này (hoặc toàn bộ nếu ít hơn)
    'EDA_SAMPLE': 200_000,        # số dòng lấy mẫu để vẽ biểu đồ EDA
    'CORR_THRESHOLD': 0.95,       # ngưỡng loại đặc trưng tương quan cao
    'MIN_CAT_FREQ': 0.005,        # category có tần suất < 0.5% trong train -> gom thành 'other'
    'MAX_CATS': 20,               # giữ tối đa 20 category phổ biến nhất mỗi cột
    'VAL_SIZE': 0.15,             # tỉ lệ validation tách từ train (UNSW-NB15)
    'RF_TUNE': True,              # có chạy RandomizedSearchCV cho Random Forest không
    'RF_TUNE_SUBSET': 40_000,     # số mẫu train dùng cho RandomizedSearchCV
    'RF_TUNE_ITER': 8,            # số cấu hình thử
    'DL_EPOCHS': 30,
    'DL_BATCH': 1024,
    'DL_PATIENCE': 5,
    'RUN_SHAP': True,
    'SHAP_SAMPLES': 200,
    'SHAP_TREES': 25,             # số cây RF dùng để tính SHAP (giảm thời gian)
}

OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('outputs')
FIG_DIR = os.path.join(OUT_DIR, 'figures')
MODEL_DIR = os.path.join(OUT_DIR, 'models')
RES_DIR = os.path.join(OUT_DIR, 'results')
for _d in (FIG_DIR, MODEL_DIR, RES_DIR):
    os.makedirs(_d, exist_ok=True)

TASKS = ['binary', 'multi']
TASK_VN = {'binary': 'Nhị phân', 'multi': 'Đa lớp'}
MODEL_NAMES = ['RandomForest', 'CNN-BiLSTM']
LABEL_COLS = ['y_bin', 'y_multi']
BIN_NAMES = {0: 'Normal', 1: 'Attack'}
BIN_COLORS = {'Normal': '#4C72B0', 'Attack': '#C44E52'}

DS_INFO = {
    'UNSW-NB15': {
        'normal': 'Normal',
        'drop_cols': ['id'],                      # cột định danh -> gây rò rỉ / làm sai việc loại trùng
        'cat_cols': ['proto', 'service', 'state'],
        'key_feats': ['sttl', 'dttl', 'dur', 'sbytes', 'dbytes', 'spkts', 'dpkts', 'rate',
                      'sload', 'smean', 'ct_state_ttl', 'ct_srv_src'],
    },
    'CIC-IDS2017': {
        'normal': 'Benign',
        'drop_cols': ['Flow ID', 'Source IP', 'Src IP', 'Destination IP', 'Dst IP', 'Timestamp',
                      'Source Port', 'Src Port', 'Label_orig'],
        'cat_cols': [],
        'key_feats': ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
                      'Total Length of Fwd Packets', 'Flow Bytes/s', 'Flow Packets/s',
                      'Fwd Packet Length Mean', 'Bwd Packet Length Max', 'Init_Win_bytes_forward',
                      'Flow IAT Mean', 'Packet Length Mean', 'Destination Port'],
    },
}

def slug(s):
    return re.sub(r'[^0-9a-zA-Z]+', '_', str(s)).strip('_').lower()

def save_fig(name):
    path = os.path.join(FIG_DIR, slug(name) + '.png')
    plt.savefig(path, bbox_inches='tight', dpi=110)
    return path

def mem_mb(df):
    return df.memory_usage(deep=True).sum() / 1024 ** 2

def reduce_memory(df):
    # Ép kiểu float64 -> float32, int64 -> int32 (nếu giá trị nằm trong khoảng int32, ngược lại -> float32)
    i32 = np.iinfo(np.int32)
    for c in df.columns:
        dt = df[c].dtype
        if dt == np.float64:
            df[c] = df[c].astype(np.float32)
        elif dt == np.int64:
            cmin, cmax = df[c].min(), df[c].max()
            ok = (cmin >= i32.min) and (cmax <= i32.max)
            df[c] = df[c].astype(np.int32 if ok else np.float32)
    return df

print('Thư mục output:', OUT_DIR)

## 2. Load dữ liệu

**Làm gì:** dò tự động mọi file `.csv`/`.parquet` trong `/kaggle/input` bằng `os.walk`, rồi nhận diện dataset dựa vào **tên cột (header)** thay vì hard-code đường dẫn:
- **UNSW-NB15**: file có cột `attack_cat` và `sttl`. Tên file chứa `train` là tập train, chứa `test` là tập test.
- **CIC-IDS2017**: file có cột `Label` và `Flow Duration` (8 file theo ngày), được gộp bằng `pd.concat`.

**Chuẩn hóa ngay khi load:**
- `strip()` tên cột (CIC-IDS2017 có khoảng trắng ở đầu tên cột, vd ` Label`).
- Ép kiểu `float64 → float32`, `int64 → int32` để giảm khoảng 50% RAM.
- Tạo 2 cột nhãn chuẩn dùng chung cho cả pipeline: `y_bin` (0 = Normal/Benign, 1 = Attack) và `y_multi` (tên lớp).

**Gộp nhãn CIC-IDS2017 (15 nhãn → 9 nhóm):**

| Nhãn gốc | Nhóm |
|---|---|
| BENIGN | Benign |
| DoS Hulk, DoS GoldenEye, DoS slowloris, DoS Slowhttptest | DoS |
| DDoS | DDoS |
| PortScan | PortScan |
| FTP-Patator, SSH-Patator | Brute Force |
| Web Attack – Brute Force, Web Attack – XSS, Web Attack – Sql Injection | Web Attack |
| Bot | Bot |
| Infiltration | Infiltration |
| Heartbleed | Heartbleed |

**Lý do gộp:**
1. Các biến thể DoS có cùng cơ chế (làm cạn tài nguyên web server) và đặc trưng luồng tương tự.
2. FTP-/SSH-Patator cùng bản chất brute-force mật khẩu.
3. Ba loại Web Attack rất ít mẫu (SQL Injection chỉ ~21 mẫu, XSS ~650), không đủ để học riêng; ở mức luồng chúng gần như không phân biệt được nếu không có payload.
4. Heartbleed (~11 mẫu) và Infiltration (~36 mẫu) được giữ riêng vì cơ chế khác hẳn. Đây là hai lớp **rất khó**, và kết quả trên chúng chỉ mang tính tham khảo vì quá ít mẫu test.

In [ ]:
# ===== 2.1 Dò tìm file dữ liệu trong /kaggle/input =====
INPUT_ROOT = '/kaggle/input'

def find_data_files(root=INPUT_ROOT):
    found = []
    for dirpath, _, fnames in os.walk(root):
        for f in fnames:
            if f.lower().endswith(('.csv', '.parquet')):
                found.append(os.path.join(dirpath, f))
    return sorted(found)

def read_header(path):
    try:
        if path.lower().endswith('.parquet'):
            import pyarrow.parquet as pq
            return [str(c).strip() for c in pq.read_schema(path).names]
        return [str(c).strip() for c in pd.read_csv(path, nrows=0, encoding='latin1').columns]
    except Exception as e:
        print('  ! Không đọc được header:', path, '|', e)
        return []

ALL_FILES = find_data_files()
print(f'Tìm thấy {len(ALL_FILES)} file CSV/Parquet trong {INPUT_ROOT}')

unsw_train, unsw_test, cic_files = [], [], []
for p in ALL_FILES:
    cols = {c.lower() for c in read_header(p)}
    name = os.path.basename(p).lower()
    if {'attack_cat', 'sttl'} <= cols:
        if 'train' in name:
            unsw_train.append(p)
        elif 'test' in name:
            unsw_test.append(p)
    elif {'label', 'flow duration'} <= cols:
        cic_files.append(p)
    elif 'sttl' in cols:
        print('  ! File có cột sttl nhưng thiếu attack_cat -> không dùng được:', p)

# CIC-IDS2017: nếu có nhiều thư mục (nhiều bản upload) -> chỉ dùng thư mục có nhiều file nhất để tránh trùng
if cic_files:
    by_dir = {}
    for p in cic_files:
        by_dir.setdefault(os.path.dirname(p), []).append(p)
    for d_, fs_ in by_dir.items():
        print(f'  Thư mục CIC-IDS2017 tìm thấy: {d_} ({len(fs_)} file)')
    pref = str(CONFIG.get('CIC_PREFERRED_DIR', '')).lower()
    preferred = [d for d in by_dir if pref and pref in d.lower()]
    best_dir = preferred[0] if preferred else max(by_dir, key=lambda d: (len(by_dir[d]), sum(os.path.getsize(x) for x in by_dir[d])))
    if len(by_dir) > 1:
        print('! Có nhiều thư mục chứa CIC-IDS2017, chỉ dùng:', best_dir)
    cic_files = sorted(by_dir[best_dir])

DATA_FILES = {}
if unsw_train and unsw_test:
    pref_u = str(CONFIG.get('UNSW_PREFERRED_DIR', '')).lower()
    def _pick(lst):
        hit = [x for x in lst if pref_u and pref_u in x.lower()]
        return (hit or lst)[0]
    tr_f = _pick(unsw_train)
    same_dir = [x for x in unsw_test if os.path.dirname(x) == os.path.dirname(tr_f)]   # test cùng thư mục với train
    te_f = same_dir[0] if same_dir else _pick(unsw_test)
    if len(unsw_train) > 1 or len(unsw_test) > 1:
        print('! Có nhiều bản UNSW-NB15:', unsw_train + unsw_test)
    DATA_FILES['UNSW-NB15'] = {'train': tr_f, 'test': te_f}
if cic_files:
    DATA_FILES['CIC-IDS2017'] = cic_files

ACTIVE = [k for k in CONFIG['DATASETS'] if k in DATA_FILES]
for k in CONFIG['DATASETS']:
    if k not in DATA_FILES:
        print(f'! KHÔNG tìm thấy {k} trong {INPUT_ROOT} -> bỏ qua dataset này (hãy Add Input nếu cần).')
assert ACTIVE, 'Không tìm thấy dataset nào! Hãy Add Input UNSW-NB15 và/hoặc CIC-IDS2017.'

print('Dataset sẽ chạy:', ACTIVE)
if 'UNSW-NB15' in DATA_FILES:
    print('UNSW-NB15:', DATA_FILES['UNSW-NB15'])
if 'CIC-IDS2017' in DATA_FILES:
    print('CIC-IDS2017:', len(cic_files), 'file')
    for p in cic_files:
        print('   ', p)

In [ ]:
# ===== 2.2 Các hàm load dữ liệu =====
def read_table(path):
    if path.lower().endswith('.parquet'):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, low_memory=False, encoding='latin1')
    df.columns = [str(c).strip() for c in df.columns]   # strip khoảng trắng tên cột
    idx_cols = [c for c in df.columns if c.lower().startswith('unnamed')]
    if idx_cols:   # cột index do to_csv() sinh ra -> là số thứ tự dòng, gây rò rỉ nếu giữ lại
        print(f'  ! Bỏ cột index thừa {idx_cols} trong {os.path.basename(path)}')
        df = df.drop(columns=idx_cols)
    return df

def load_unsw(files):
    tr, te = read_table(files['train']), read_table(files['test'])
    if len(tr) < len(te):   # một số bản upload trên Kaggle bị đặt ngược tên train/test
        print('! Phát hiện tên file train/test bị đảo (train nhỏ hơn test) -> hoán đổi lại.')
        tr, te = te, tr
    out = []
    for df in (tr, te):
        df.columns = [c.lower() for c in df.columns]
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = df[c].astype(str).str.strip()
        cat = df['attack_cat'].astype(str).str.strip().replace({'nan': 'Normal', '': 'Normal', 'None': 'Normal', 'Backdoors': 'Backdoor'})
        if not (cat == 'Normal').any():   # attack_cat đã bị mã hóa số -> đặt lại tên lớp bình thường
            norm_val = cat[df['label'].astype(int).to_numpy() == 0].mode().iloc[0]
            print(f'  ! attack_cat không có giá trị Normal -> coi giá trị {norm_val} là Normal')
            cat = cat.replace({norm_val: 'Normal'})
        df['y_multi'] = cat.values
        df['y_bin'] = df['label'].astype(int).astype(np.int8).values
        df.drop(columns=['attack_cat', 'label'], inplace=True)
        out.append(reduce_memory(df))
    return out[0], out[1]

def normalize_cic_label(s):
    # Chuẩn hóa ký tự lỗi encoding trong nhãn Web Attack (vd 'Web Attack � Brute Force')
    s = str(s).strip()
    if s.lower().startswith('web attack'):
        rest = re.sub(r'^web attack[^a-z]*', '', s, flags=re.I)
        return 'Web Attack - ' + rest.strip()
    return s

def group_cic_label(lbl):
    l = str(lbl).lower()
    if l == 'benign':
        return 'Benign'
    if 'ddos' in l:
        return 'DDoS'
    if l.startswith('dos'):
        return 'DoS'
    if 'portscan' in l or 'port scan' in l:
        return 'PortScan'
    if 'web attack' in l or 'xss' in l or 'sql' in l:
        return 'Web Attack'
    if 'patator' in l or 'brute' in l:
        return 'Brute Force'
    if l.startswith('bot'):
        return 'Bot'
    if 'infiltration' in l:
        return 'Infiltration'
    if 'heartbleed' in l:
        return 'Heartbleed'
    return 'Other'

def load_cic(files):
    dfs = []
    for p in files:
        d = reduce_memory(read_table(p))
        print(f'  - {os.path.basename(p):<75s} {str(d.shape):>15s}')
        dfs.append(d)
    df = pd.concat(dfs, ignore_index=True)
    del dfs; gc.collect()
    lab_col = [c for c in df.columns if c.lower() == 'label'][0]
    keep_text = set(DS_INFO['CIC-IDS2017']['drop_cols']) | {lab_col}
    for c in df.columns:   # cột số bị đọc thành chuỗi (vd 'Infinity') -> ép về số
        if c not in keep_text and not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df = reduce_memory(df)
    lab_str = df[lab_col].astype(str).str.strip()
    valid = df[lab_col].notna() & ~lab_str.str.lower().isin(['', 'nan', 'none'])
    if (~valid).any():   # vd bản TrafficLabelling: file Thursday-WebAttacks có rất nhiều dòng rỗng ở cuối
        print(f'  ! Bỏ {int((~valid).sum()):,} dòng không có nhãn (dòng rỗng)')
        df = df.loc[valid].reset_index(drop=True)
        lab_str = lab_str.loc[valid].reset_index(drop=True)
    raw = lab_str
    norm_map = {u: normalize_cic_label(u) for u in raw.unique()}
    df['Label_orig'] = raw.map(norm_map).values
    group_map = {v: group_cic_label(v) for v in set(norm_map.values())}
    df['y_multi'] = df['Label_orig'].map(group_map).values
    df['y_bin'] = (df['y_multi'] != 'Benign').astype(np.int8).values
    df.drop(columns=[lab_col], inplace=True)
    return df

In [ ]:
# ===== 2.3 Load dữ liệu & thông tin tổng quan =====
RAW = {}
t0 = time.time()
if 'UNSW-NB15' in ACTIVE:
    print('Đang load UNSW-NB15 ...')
    _tr, _te = load_unsw(DATA_FILES['UNSW-NB15'])
    RAW['UNSW-NB15'] = {'train': _tr, 'test': _te}
    del _tr, _te
if 'CIC-IDS2017' in ACTIVE:
    print('Đang load CIC-IDS2017 ...')
    RAW['CIC-IDS2017'] = {'full': load_cic(DATA_FILES['CIC-IDS2017'])}
gc.collect()
print(f'Load xong trong {time.time() - t0:.1f}s')

for key, parts in RAW.items():
    for part, df in parts.items():
        print('=' * 100)
        print(f'{key} [{part}]  shape = {df.shape}  |  memory = {mem_mb(df):.1f} MB')
        print('dtypes:', {str(k): int(v) for k, v in df.dtypes.astype(str).value_counts().items()})
        display(df.head())

**Nhận xét (đối chiếu với số liệu in ra ở trên):**
- **UNSW-NB15**: bản chính thức có 175.341 dòng train và 82.332 dòng test, 45 cột (gồm `id`, 3 cột phân loại `proto`/`service`/`state`, ~39 cột số, cùng 2 cột nhãn đã chuyển thành `y_bin`/`y_multi`). Dữ liệu nhỏ, chiếm chỉ vài chục MB RAM sau khi ép kiểu.
- **CIC-IDS2017**: ~2,83 triệu dòng và ~79 cột, hầu như toàn bộ là số. Sau khi ép về `float32`/`int32`, bộ nhớ khoảng 1 GB, vừa đủ thoải mái với 30 GB RAM của Kaggle. Lưu ý cột `Fwd Header Length.1` là bản sao của `Fwd Header Length` (lỗi của CICFlowMeter), sẽ bị loại ở bước lọc tương quan.
- Các cột tên đã được `strip()`, nên mọi tham chiếu cột phía sau (vd `Flow Duration`) không bị lỗi khoảng trắng.

## 3. EDA – Phân tích khám phá dữ liệu

**Làm gì:** khảo sát chất lượng dữ liệu (thiếu, inf, trùng lặp), phân bố nhãn, đặc trưng phân loại, phân phối đặc trưng số, tương quan, khác biệt giữa Normal và Attack, cùng các cột hằng số.

**Tại sao:** mỗi vấn đề phát hiện ở đây quyết định một bước tiền xử lý ở mục 4. Ví dụ: có inf thì phải thay bằng NaN, có trùng lặp thì phải loại trước khi chia tập, mất cân bằng lớp thì cần class weight và metric F1-macro.

Với UNSW-NB15, EDA thực hiện trên **train + test gộp** (có cột `split`) để thấy cả độ lệch phân bố giữa 2 tập. Các biểu đồ dùng mẫu ngẫu nhiên `EDA_SAMPLE` dòng để vẽ nhanh, còn các thống kê đếm (NaN, inf, trùng lặp, nhãn) tính trên **toàn bộ** dữ liệu.

In [ ]:
# ===== Hàm tiện ích cho EDA =====
def feature_cols(df, key):
    excl = set(DS_INFO[key]['drop_cols']) | set(LABEL_COLS) | {'split', 'Label_orig'}
    return [c for c in df.columns if c not in excl]

def numeric_cols(df, key):
    return [c for c in feature_cols(df, key) if pd.api.types.is_numeric_dtype(df[c])]

def signed_log(x):
    # log1p có dấu: giữ được giá trị âm, nén các đặc trưng lệch phải rất mạnh (bytes, packets, duration...)
    return np.sign(x) * np.log1p(np.abs(x))

def eda_sample(df, n=None):
    n = n or CONFIG['EDA_SAMPLE']
    return df.sample(min(len(df), n), random_state=SEED)

def pick_features(df, key, k=8):
    # Lấy các đặc trưng quan trọng khai báo trong DS_INFO (không phân biệt hoa thường); thiếu thì bù bằng cột phương sai lớn
    lower = {c.lower(): c for c in df.columns}
    feats = [lower[f.lower()] for f in DS_INFO[key]['key_feats'] if f.lower() in lower]
    feats = [f for f in feats if pd.api.types.is_numeric_dtype(df[f])]
    if len(feats) < k:
        rest = [c for c in numeric_cols(df, key) if c not in feats]
        s = df[rest].sample(min(len(df), 100_000), random_state=SEED).astype('float64')
        var = s.replace([np.inf, -np.inf], np.nan).var().sort_values(ascending=False)
        feats += var.index.tolist()
    return feats[:k]

def rows_hash(df, cols):
    # Hash từng dòng (đưa số về float64, chuỗi về str để hash nhất quán giữa các tập)
    tmp = df[cols].copy()
    for c in tmp.columns:
        tmp[c] = tmp[c].astype('float64') if pd.api.types.is_numeric_dtype(tmp[c]) else tmp[c].astype(str)
    return pd.util.hash_pandas_object(tmp, index=False).to_numpy()

EDA = {}
for key in ACTIVE:
    if key == 'UNSW-NB15':
        EDA[key] = pd.concat([RAW[key]['train'].assign(split='train'),
                              RAW[key]['test'].assign(split='test')], ignore_index=True)
    else:
        EDA[key] = RAW[key]['full']
    print(f'{key}: dữ liệu dùng cho EDA có shape = {EDA[key].shape}')

### 3.1 Thống kê mô tả & chất lượng dữ liệu
Đếm số ô thiếu (NaN), số giá trị vô cực (inf), số dòng trùng lặp theo 3 cách: có cột định danh, bỏ cột định danh, chỉ xét đặc trưng (để phát hiện **nhãn mâu thuẫn**). Với UNSW-NB15 kiểm tra thêm **số bản ghi test trùng với train** (nguy cơ rò rỉ).

In [ ]:
def quality_report(df, key):
    feats = feature_cols(df, key)
    num = numeric_cols(df, key)
    id_present = [c for c in DS_INFO[key]['drop_cols'] if c in df.columns and c != 'Label_orig']
    print('=' * 100)
    print(f'{key}: {df.shape[0]:,} dòng | {len(feats)} đặc trưng ({len(num)} số, {len(feats) - len(num)} phân loại) | cột định danh: {id_present}')
    nan_cnt = df[feats].isna().sum()
    inf_cnt = pd.Series({c: int(np.isinf(df[c].to_numpy()).sum()) for c in num
                         if pd.api.types.is_float_dtype(df[c])}, dtype='int64')
    print(f'- Tổng số ô thiếu (NaN): {int(nan_cnt.sum()):,} | số cột có NaN: {int((nan_cnt > 0).sum())}')
    print(f'- Tổng số giá trị inf  : {int(inf_cnt.sum()):,} | số cột có inf: {int((inf_cnt > 0).sum())}')
    bad = pd.DataFrame({'NaN': nan_cnt, 'inf': inf_cnt}).fillna(0).astype(int)
    bad = bad[(bad['NaN'] > 0) | (bad['inf'] > 0)]
    if len(bad):
        display(bad.sort_values(['NaN', 'inf'], ascending=False))
    dup_with_id = int(df[feats + id_present + LABEL_COLS].duplicated().sum())
    dup_full = int(df[feats + LABEL_COLS].duplicated().sum())
    dup_feat = int(df[feats].duplicated().sum())
    print(f'- Dòng trùng lặp (tính cả cột định danh)          : {dup_with_id:,}')
    print(f'- Dòng trùng lặp (bỏ cột định danh, đặc trưng+nhãn): {dup_full:,} ({dup_full / len(df) * 100:.2f}%)')
    print(f'- Dòng trùng đặc trưng nhưng KHÁC nhãn (mâu thuẫn) : {dup_feat - dup_full:,}')
    if 'split' in df.columns:
        h = rows_hash(df, feats)
        is_tr = df['split'].to_numpy() == 'train'
        ov = int(np.isin(h[~is_tr], h[is_tr]).sum())
        print(f'- Bản ghi test có đặc trưng trùng với train       : {ov:,} ({ov / (~is_tr).sum() * 100:.2f}% tập test)')
    desc = df[num].describe().T
    display(desc.round(3))

for key in ACTIVE:
    quality_report(EDA[key], key)

**Nhận xét (đối chiếu với số liệu ở trên):**
- **Cột `id` của UNSW-NB15** làm mọi dòng trở thành duy nhất: số dòng trùng khi tính cả `id` bằng 0, nhưng khi bỏ `id` lại xuất hiện **rất nhiều dòng trùng** (tập trung ở các lớp Generic, Exploits, DoS). Vì vậy phải **bỏ `id` trước rồi mới loại trùng**. Nếu giữ `id`, mô hình còn có thể học thứ tự bản ghi (rò rỉ).
- UNSW-NB15 có một phần không nhỏ bản ghi test **trùng hoàn toàn đặc trưng với train**. Nếu giữ chúng ở cả hai tập, kết quả test bị thổi phồng, nên các bản ghi train này sẽ bị loại ở mục 4.
- **CIC-IDS2017** có NaN (chủ yếu ở `Flow Bytes/s`, khoảng 1–2 nghìn dòng) và **inf** ở `Flow Bytes/s`, `Flow Packets/s` (do chia cho `Flow Duration = 0`). Số này rất nhỏ so với 2,8 triệu dòng. Ngoài ra có khoảng 300 nghìn dòng trùng lặp (~10%).
- Thống kê `describe()` cho thấy nhiều đặc trưng có **max lớn hơn median hàng nghìn lần** (bytes, packets, duration, rate): phân phối lệch phải rất mạnh, nên cần biến đổi log trước khi chuẩn hóa (quan trọng với mạng nơ-ron). CIC-IDS2017 còn có một số giá trị **âm vô lý** (vd `Init_Win_bytes_forward = -1`, `Flow Duration` âm), là dấu hiệu lỗi của công cụ trích xuất. Ta dùng signed-log để giữ được dấu.

### 3.2 Phân bố nhãn (nhị phân & đa lớp)

In [ ]:
def plot_label_distribution(df, key):
    vc_bin = df['y_bin'].map(BIN_NAMES).value_counts().reindex(['Normal', 'Attack']).fillna(0)
    vc_multi = df['y_multi'].value_counts()
    fig, axes = plt.subplots(1, 3, figsize=(21, 5.5), gridspec_kw={'width_ratios': [1, 1, 2.3]})
    bin_colors = [BIN_COLORS[i] for i in vc_bin.index]
    bars = axes[0].bar(vc_bin.index, vc_bin.values, color=bin_colors)
    axes[0].bar_label(bars, labels=[f'{int(v):,}' for v in vc_bin.values])
    axes[0].set_title(f'{key} – Phân bố nhãn nhị phân')
    axes[0].set_xlabel('Nhãn'); axes[0].set_ylabel('Số mẫu')
    axes[1].pie(vc_bin.values, labels=vc_bin.index, autopct='%1.1f%%', startangle=90,
                colors=bin_colors, wedgeprops={'edgecolor': 'white'})
    axes[1].set_title(f'{key} – Tỉ lệ Normal / Attack')
    colors = ['#4C72B0' if c == DS_INFO[key]['normal'] else '#C44E52' for c in vc_multi.index]
    bars = axes[2].bar(vc_multi.index.astype(str), vc_multi.values, color=colors)
    axes[2].bar_label(bars, labels=[f'{v:,}' for v in vc_multi.values], fontsize=8, rotation=90, padding=2)
    axes[2].set_yscale('log')
    axes[2].set_title(f'{key} – Phân bố nhãn đa lớp (thang log)')
    axes[2].set_xlabel('Lớp'); axes[2].set_ylabel('Số mẫu (log)')
    axes[2].tick_params(axis='x', rotation=45)
    plt.tight_layout(); save_fig(f'eda_label_distribution_{key}'); plt.show()

    tbl = pd.DataFrame({'Số mẫu': vc_multi, 'Tỉ lệ (%)': (vc_multi / vc_multi.sum() * 100).round(3)})
    display(tbl)
    print(f'{key}: tỉ lệ mất cân bằng (lớp lớn nhất / lớp nhỏ nhất) = {vc_multi.max() / vc_multi.min():,.1f} lần')
    if 'split' in df.columns:
        print('Phân bố lớp theo tập train/test gốc:')
        display(pd.crosstab(df['y_multi'], df['split'], margins=True))
    if 'Label_orig' in df.columns:
        orig = df.groupby('Label_orig')['y_multi'].agg(['first', 'size'])
        orig.columns = ['Nhóm sau khi gộp', 'Số mẫu']
        print('Nhãn gốc CIC-IDS2017 và nhóm sau khi gộp:')
        display(orig.sort_values('Số mẫu', ascending=False))

for key in ACTIVE:
    plot_label_distribution(EDA[key], key)

**Nhận xét – mất cân bằng lớp:**
- **UNSW-NB15**: ở mức nhị phân khá cân bằng. Điểm đặc biệt là trong tập train chính thức, Attack chiếm đa số (~68%), còn ở tập test khoảng 55%. Ở mức đa lớp lại **mất cân bằng nặng**: Generic và Exploits chiếm phần lớn tấn công, trong khi Worms chỉ ~170 mẫu, Shellcode/Backdoor/Analysis chỉ vài nghìn, tức chênh lệch lớp lớn nhất/nhỏ nhất vào khoảng hàng trăm lần.
- **CIC-IDS2017**: Benign chiếm ~80%. Sau khi gộp nhãn vẫn còn mất cân bằng **cực đoan**: DoS, DDoS, PortScan có hàng trăm nghìn mẫu, còn Infiltration (~36) và Heartbleed (~11) chênh lệch tới **hàng trăm nghìn lần**.
- **Hệ quả:** Accuracy không phản ánh đúng chất lượng. Một mô hình đoán toàn Benign đã đạt ~80% Accuracy trên CIC-IDS2017. Vì vậy cần ưu tiên **F1-macro, Recall lớp tấn công (Detection Rate) và FAR**, đồng thời dùng **class_weight** khi huấn luyện. Các lớp siêu hiếm (Heartbleed, Infiltration, Worms) sẽ có metric dao động lớn vì tập test chỉ chứa vài mẫu.

### 3.3 Phân tích đặc trưng phân loại
UNSW-NB15 có 3 cột phân loại `proto`, `service`, `state`. CIC-IDS2017 không có cột chuỗi, nên ta xem `Destination Port` (và `Protocol` nếu có) như biến phân loại để khảo sát. Mỗi biểu đồ hiển thị top 10 giá trị, các giá trị còn lại gom vào `other`.

In [ ]:
def plot_categorical(df, key, top=10):
    cols = [c for c in DS_INFO[key]['cat_cols'] if c in df.columns]
    if not cols:   # CIC-IDS2017: xem Destination Port / Protocol như biến phân loại
        cols = [c for c in df.columns if c.lower() in ('destination port', 'dst port', 'protocol')]
    if not cols:
        print(f'{key}: không có đặc trưng phân loại để vẽ.')
        return
    s = eda_sample(df)
    lab = s['y_bin'].map(BIN_NAMES).values
    fig, axes = plt.subplots(1, len(cols), figsize=(7 * len(cols), 5.5), squeeze=False)
    for ax, c in zip(axes.ravel(), cols):
        vals = s[c].astype(str)
        top_vals = vals.value_counts().index[:top]
        tmp = pd.DataFrame({c: vals.where(vals.isin(top_vals), 'other').values, 'Nhãn': lab})
        sns.countplot(data=tmp, x=c, hue='Nhãn', order=tmp[c].value_counts().index,
                      hue_order=['Normal', 'Attack'], palette=BIN_COLORS, ax=ax)
        ax.set_yscale('log')
        ax.set_title(f'{key} – {c} (top {top}) theo nhãn')
        ax.set_xlabel(c); ax.set_ylabel('Số mẫu (log)')
        ax.tick_params(axis='x', rotation=45)
        print(f'{key} – {c}: {df[c].nunique():,} giá trị khác nhau')
    plt.tight_layout(); save_fig(f'eda_categorical_{key}'); plt.show()
    for c in cols:
        ct = pd.crosstab(s[c].astype(str), s['y_bin'].map(BIN_NAMES))
        ct['Tổng'] = ct.sum(axis=1)
        if 'Attack' in ct.columns:
            ct['% Attack'] = (ct['Attack'] / ct['Tổng'] * 100).round(1)
        print(f'Tỉ lệ Attack theo {c} (top {top} giá trị phổ biến nhất):')
        display(ct.sort_values('Tổng', ascending=False).head(top))

for key in ACTIVE:
    plot_categorical(EDA[key], key)

**Nhận xét:**
- **UNSW-NB15 – `proto`** có hơn 130 giá trị, nhưng `tcp`, `udp` chiếm phần lớn. Rất nhiều giao thức hiếm (vd `sctp`, `any`, `gre`…) **chỉ xuất hiện ở lưu lượng tấn công**. Đây là tín hiệu mạnh, nhưng cũng dễ gây overfit, nên các category hiếm (< 0,5%) sẽ được gom thành `other` trước khi One-Hot.
- **`service`**: giá trị `-` (không xác định) chiếm đa số. `dns` gắn chặt với lớp Generic, còn `http`/`ftp` xuất hiện ở cả hai nhãn.
- **`state`**: `FIN` chủ yếu là Normal, trong khi `INT` và `CON` lệch mạnh về Attack. Đây là đặc trưng phân biệt tốt.
- **CIC-IDS2017 – `Destination Port`**: cổng 80/443/53 chiếm đa số. Tấn công DoS/DDoS/Web Attack tập trung vào cổng 80, Brute Force vào 21/22. Cổng đích có giá trị phân loại cao, nhưng mô hình có thể **phụ thuộc vào cấu hình testbed** (vd luôn tấn công cổng 80), nên cần lưu ý khi tổng quát hóa sang mạng thật. Ta giữ cột này ở dạng số như bản gốc của dataset.

### 3.4 Phân phối đặc trưng số (histogram/KDE & boxplot)
Với đặc trưng lệch mạnh (|skewness| > 2), trục x được vẽ trên thang `log1p` có dấu. Boxplot luôn dùng thang log để thấy rõ outlier. Bảng cuối cùng cho biết tỉ lệ outlier theo quy tắc IQR (ngoài khoảng [Q1 − 1,5·IQR, Q3 + 1,5·IQR]).

In [ ]:
def plot_numeric_distributions(df, key, k=8):
    feats = pick_features(df, key, k)
    s = eda_sample(df)
    lab = s['y_bin'].map(BIN_NAMES).values
    ncol = 4
    nrow = int(np.ceil(len(feats) / ncol))
    # --- Histogram + KDE theo nhãn ---
    fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.8 * nrow), squeeze=False)
    axes = axes.ravel()
    for ax, f in zip(axes, feats):
        x = s[f].astype('float64').replace([np.inf, -np.inf], np.nan)
        skew = x.skew()
        use_log = bool(abs(skew) > 2)
        xv = signed_log(x) if use_log else x
        tmp = pd.DataFrame({'v': xv.values, 'Nhãn': lab}).dropna()
        sns.histplot(data=tmp, x='v', hue='Nhãn', hue_order=['Normal', 'Attack'], palette=BIN_COLORS,
                     bins=50, stat='density', common_norm=False, element='step', kde=True, ax=ax)
        ax.set_title(f'{f} (skew = {skew:.1f})', fontsize=10)
        ax.set_xlabel(('log1p(' + f + ')') if use_log else f)
        ax.set_ylabel('Mật độ')
    for ax in axes[len(feats):]:
        ax.axis('off')
    fig.suptitle(f'{key} – Phân phối đặc trưng số theo nhãn (mẫu {len(s):,} dòng)', fontsize=14, y=1.01)
    plt.tight_layout(); save_fig(f'eda_numeric_hist_{key}'); plt.show()

    # --- Boxplot (thang log) để phát hiện outlier ---
    fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.8 * nrow), squeeze=False)
    axes = axes.ravel()
    out = {}
    for ax, f in zip(axes, feats):
        x = s[f].astype('float64').replace([np.inf, -np.inf], np.nan)
        tmp = pd.DataFrame({'v': signed_log(x).values, 'Nhãn': lab}).dropna()
        sns.boxplot(data=tmp, x='Nhãn', y='v', order=['Normal', 'Attack'], palette=BIN_COLORS,
                    fliersize=1, ax=ax)
        ax.set_title(f, fontsize=10); ax.set_xlabel('Nhãn'); ax.set_ylabel('log1p(giá trị)')
        q1, q3 = x.quantile(0.25), x.quantile(0.75)
        iqr = q3 - q1
        out[f] = float(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).mean() * 100)
    for ax in axes[len(feats):]:
        ax.axis('off')
    fig.suptitle(f'{key} – Boxplot đặc trưng số (thang log)', fontsize=14, y=1.01)
    plt.tight_layout(); save_fig(f'eda_numeric_box_{key}'); plt.show()
    display(pd.Series(out, name='% outlier (IQR)').round(2).to_frame())

for key in ACTIVE:
    plot_numeric_distributions(EDA[key], key)

**Nhận xét:**
- Hầu hết đặc trưng lưu lượng (bytes, packets, duration, load, rate) có **skewness rất lớn** (thường > 10). Trên thang gốc, gần như toàn bộ mẫu dồn sát 0. Sau khi lấy log, phân phối dễ nhìn hơn và thường có **nhiều đỉnh**, phản ánh nhiều kiểu lưu lượng khác nhau.
- **UNSW-NB15**: `sttl`/`dttl` phân tách Normal và Attack rất rõ (tấn công tập trung ở các giá trị TTL đặc thù như 254 hoặc 62). Đây là “dấu vân tay” của testbed và có thể khiến mô hình đạt điểm rất cao trên dataset này nhưng khó tổng quát hóa. `sbytes`, `smean` của Attack thường nhỏ và tập trung hơn.
- **CIC-IDS2017**: `Flow Duration`, `Flow IAT Mean` của DoS thường dài (slowloris/Slowhttptest giữ kết nối lâu), PortScan có rất ít gói và byte, còn `Init_Win_bytes_forward` khác biệt rõ giữa các loại công cụ tấn công.
- **Outlier** theo IQR chiếm tỉ lệ đáng kể (có cột > 10%). Đây là **outlier tự nhiên** của lưu lượng mạng (luồng tải file lớn, flood…) và thường chính là tín hiệu tấn công, nên **không loại bỏ**. Thay vào đó ta dùng biến đổi signed-log cùng StandardScaler để giảm ảnh hưởng của outlier tới mạng nơ-ron. Random Forest vốn không nhạy với outlier.

### 3.5 Ma trận tương quan
Tính hệ số tương quan Pearson trên mẫu dữ liệu, sau đó liệt kê các cặp có |r| > 0,95. Ở mục 4, các cột tương quan cao sẽ được loại, nhưng ngưỡng **chỉ được tính trên tập train** để tránh rò rỉ.

In [ ]:
def correlation_analysis(df, key, thr=None):
    thr = thr or CONFIG['CORR_THRESHOLD']
    num = numeric_cols(df, key)
    s = eda_sample(df)[num].astype('float64').replace([np.inf, -np.inf], np.nan)
    s = s.loc[:, s.nunique() > 1]
    corr = s.corr()
    n = len(corr)
    plt.figure(figsize=(max(12, 0.28 * n + 4), max(10, 0.26 * n + 3)))
    sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, square=True,
                xticklabels=True, yticklabels=True, cbar_kws={'shrink': 0.6, 'label': 'Pearson r'})
    plt.xticks(fontsize=6); plt.yticks(fontsize=6)
    plt.title(f'{key} – Ma trận tương quan Pearson ({n} đặc trưng số)', fontsize=13)
    plt.xlabel('Đặc trưng'); plt.ylabel('Đặc trưng')
    plt.tight_layout(); save_fig(f'eda_correlation_{key}'); plt.show()
    a = corr.abs().to_numpy()
    iu = np.triu_indices_from(a, k=1)
    pairs = pd.DataFrame({'feat_1': corr.index.to_numpy()[iu[0]], 'feat_2': corr.columns.to_numpy()[iu[1]], 'abs_corr': a[iu]})
    pairs = pairs[pairs['abs_corr'] > thr].sort_values('abs_corr', ascending=False).reset_index(drop=True)
    print(f'{key}: {len(pairs)} cặp đặc trưng có |r| > {thr}')
    display(pairs.head(40).round(4))
    return pairs

CORR_PAIRS_EDA = {key: correlation_analysis(EDA[key], key) for key in ACTIVE}

**Nhận xét:**
- **UNSW-NB15** có một số cặp gần như trùng thông tin: `sbytes`–`sloss`, `dbytes`–`dloss`, `spkts`–`sbytes`, `ct_srv_src`–`ct_srv_dst`, `ct_dst_src_ltm`–`ct_srv_*`, `swin`–`dwin`, `is_ftp_login`–`ct_ftp_cmd`… Đây là các đặc trưng được dẫn xuất lẫn nhau.
- **CIC-IDS2017** có **rất nhiều** cặp tương quan > 0,95, thậm chí bằng 1: `Fwd Header Length` với `Fwd Header Length.1` (cột trùng), `Subflow Fwd Packets` với `Total Fwd Packets`, `Avg Fwd Segment Size` với `Fwd Packet Length Mean`, các nhóm `Packet Length Mean/Std/Variance`, `Flow IAT Max`–`Idle Max`… Lý do là CICFlowMeter tính nhiều thống kê dư thừa từ cùng một nguồn.
- **Hướng xử lý:** loại một cột trong mỗi cặp |r| > 0,95 để giảm số chiều, giảm thời gian huấn luyện và giảm đa cộng tuyến. Việc này gần như không làm mất thông tin, vì cột bị loại gần như là hàm tuyến tính của cột còn lại.

### 3.6 So sánh đặc trưng quan trọng giữa Normal và Attack
Bên trái là median (thang log) của các đặc trưng chính theo nhãn nhị phân. Bên phải là heatmap median theo **từng lớp tấn công**, chuẩn hóa min-max theo cột để so sánh được giữa các đặc trưng; số trong ô là `log1p(median)`.

In [ ]:
def compare_normal_attack(df, key, k=10):
    feats = pick_features(df, key, k)
    s = df[feats].astype('float64').replace([np.inf, -np.inf], np.nan)
    grp = df['y_bin'].map(BIN_NAMES).to_numpy()
    med = s.groupby(grp).median().T[['Normal', 'Attack']]
    mean = s.groupby(grp).mean().T[['Normal', 'Attack']]
    tbl = pd.concat({'Median': med, 'Mean': mean}, axis=1)
    display(tbl.round(2))

    fig, axes = plt.subplots(1, 2, figsize=(22, 6.5), gridspec_kw={'width_ratios': [1, 1.3]})
    signed_log(med).plot.bar(ax=axes[0], color=[BIN_COLORS['Normal'], BIN_COLORS['Attack']])
    axes[0].set_title(f'{key} – Median đặc trưng: Normal vs Attack (thang log)')
    axes[0].set_xlabel('Đặc trưng'); axes[0].set_ylabel('log1p(median)')
    axes[0].tick_params(axis='x', rotation=60)
    axes[0].legend(title='Nhãn')

    z = signed_log(s.groupby(df['y_multi'].to_numpy()).median())
    rng = (z.max() - z.min()).replace(0, 1)
    zn = (z - z.min()) / rng
    sns.heatmap(zn, annot=z.to_numpy(), fmt='.1f', annot_kws={'fontsize': 7}, cmap='YlOrRd', ax=axes[1],
                cbar_kws={'label': 'median (log) chuẩn hóa theo cột'})
    axes[1].set_title(f'{key} – log1p(median) theo từng lớp')
    axes[1].set_xlabel('Đặc trưng'); axes[1].set_ylabel('Lớp')
    axes[1].tick_params(axis='x', rotation=60)
    plt.tight_layout(); save_fig(f'eda_normal_vs_attack_{key}'); plt.show()

for key in ACTIVE:
    compare_normal_attack(EDA[key], key)

**Nhận xét:**
- **UNSW-NB15**: median `sttl` của Attack cao vượt trội so với Normal (254 so với 31). `ct_state_ttl` cũng khác biệt rõ. Các lớp Generic, Exploits, DoS, Reconnaissance có median `dur`/`sbytes` rất nhỏ (luồng ngắn), trong khi Normal có lưu lượng hai chiều (`dbytes`, `dpkts`) lớn hơn hẳn. Heatmap cho thấy **DoS, Exploits, Analysis, Backdoor có hồ sơ median gần giống nhau**, gợi ý trước rằng mô hình sẽ hay nhầm giữa các lớp này.
- **CIC-IDS2017**: mỗi nhóm tấn công có “chữ ký thống kê” riêng. PortScan có ít gói, ít byte và duration rất ngắn. DoS có duration và IAT dài. DDoS có `Bwd Packet Length Max` lớn. Brute Force có kích thước gói đặc trưng. Bot và Infiltration lại có hồ sơ **gần với Benign**, nên sẽ khó phát hiện.
- Sự khác biệt rõ rệt này giải thích vì sao các mô hình cây thường đạt điểm rất cao trên cả hai dataset, đặc biệt là CIC-IDS2017.

### 3.7 Đặc trưng hằng số / gần hằng số
Cột có phương sai bằng 0 không mang thông tin và cần loại. Cột gần hằng số (một giá trị chiếm ≥ 99,9%) được liệt kê để tham khảo; ta vẫn giữ chúng vì phần 0,1% còn lại có thể đúng là tấn công hiếm.

In [ ]:
def constant_features(df, key):
    feats = feature_cols(df, key)
    nun = df[feats].nunique(dropna=False)
    const = nun[nun <= 1].index.tolist()
    s = eda_sample(df, 500_000)
    quasi = {}
    for c in feats:
        if c in const:
            continue
        top = s[c].value_counts(normalize=True, dropna=False).iloc[0]
        if top >= 0.999:
            quasi[c] = round(float(top) * 100, 3)
    print(f'{key}: {len(const)} cột HẰNG SỐ (phương sai = 0): {const}')
    print(f'{key}: {len(quasi)} cột GẦN hằng số (1 giá trị chiếm >= 99.9%)')
    if quasi:
        display(pd.Series(quasi, name='% giá trị phổ biến nhất').sort_values(ascending=False).to_frame())
    return const

CONST_EDA = {key: constant_features(EDA[key], key) for key in ACTIVE}

**Nhận xét:**
- **CIC-IDS2017** có khoảng 8 cột hằng số (toàn 0): `Bwd PSH Flags`, `Bwd URG Flags`, `Fwd Avg Bytes/Bulk`, `Fwd Avg Packets/Bulk`, `Fwd Avg Bulk Rate`, `Bwd Avg Bytes/Bulk`, `Bwd Avg Packets/Bulk`, `Bwd Avg Bulk Rate`. Nguyên nhân là CICFlowMeter không tính được các trường này, nên chúng **bắt buộc phải loại**. Ngoài ra có một số cột cờ gần hằng số như `Fwd URG Flags`, `CWE Flag Count`, `RST Flag Count`.
- **UNSW-NB15** thường không có cột hằng số tuyệt đối. Các cột như `is_ftp_login`, `ct_ftp_cmd`, `is_sm_ips_ports` gần hằng số nhưng mang thông tin về tấn công cụ thể, nên được giữ lại.
- Việc loại cột hằng số ở mục 4 sẽ được tính **lại trên tập train** cho đúng quy trình.

### 3.8 Kết luận EDA – vấn đề dữ liệu & hướng xử lý

| # | Vấn đề | Dataset | Hướng xử lý (mục 4) |
|---|---|---|---|
| 1 | Cột định danh/rò rỉ (`id`; nếu có thì `Flow ID`, IP, `Timestamp`, cổng nguồn) | Cả hai | Loại **trước** khi tìm trùng lặp |
| 2 | Giá trị `inf` (chia cho duration = 0) | CIC-IDS2017 | `inf/-inf → NaN` |
| 3 | Giá trị thiếu NaN (< 0,1% số dòng) | CIC-IDS2017 | **Xóa dòng** vì tỉ lệ rất nhỏ, không đáng impute. Pipeline vẫn có impute median (học trên train) cho dữ liệu mới khi triển khai |
| 4 | Trùng lặp lớn; bản ghi test trùng train | Cả hai | `drop_duplicates()` **trước khi chia tập**; với UNSW-NB15 loại thêm bản ghi train trùng với test |
| 5 | Cột hằng số | CIC-IDS2017 | Loại (tính trên train) |
| 6 | Nhiều cặp tương quan > 0,95 | Cả hai | Loại một cột mỗi cặp (tính trên train) |
| 7 | Đặc trưng phân loại nhiều category hiếm (`proto` > 130 giá trị) | UNSW-NB15 | Gom category < 0,5% thành `other` rồi One-Hot |
| 8 | Phân phối lệch mạnh, nhiều outlier tự nhiên | Cả hai | Không xóa outlier; dùng signed-log + StandardScaler (fit trên train) |
| 9 | Mất cân bằng lớp nghiêm trọng | Cả hai | `class_weight` (RF: balanced; DL: sqrt-balanced); đánh giá bằng F1-macro, DR, FAR |
| 10 | Dữ liệu lớn (~2,8 triệu dòng) | CIC-IDS2017 | Lấy mẫu phân tầng 50% sau khi loại trùng, giữ trọn lớp hiếm |

In [ ]:
# Giải phóng bộ nhớ của EDA (dữ liệu gốc vẫn nằm trong RAW)
del EDA
gc.collect();

## 4. Tiền xử lý dữ liệu

**Thứ tự xử lý được thiết kế để chống rò rỉ dữ liệu:**
1. Loại cột định danh/rò rỉ (`id`, `Flow ID`, IP, `Timestamp`, cổng nguồn).
2. `inf/-inf → NaN`. Nếu số dòng chứa NaN < 1% thì **xóa dòng**; ngược lại để bước 6 impute median.
3. **Loại trùng lặp TRƯỚC khi chia tập**, để cùng một bản ghi không thể nằm cả trong train lẫn test. Với UNSW-NB15 (split gốc có sẵn), ta loại trùng trong từng tập, rồi **xóa khỏi train các bản ghi có đặc trưng trùng với test**. Tập test chính thức được giữ nguyên, chỉ bỏ dòng lặp.
4. (CIC-IDS2017) Lấy mẫu phân tầng giữ trọn lớp hiếm, rồi chia **stratified 70/15/15**. UNSW-NB15 tách 15% train làm validation.
5. Mã hóa nhãn: `LabelEncoder` cho nhãn đa lớp; nhãn nhị phân dùng trực tiếp 0/1.
6. `NIDSPreprocessor` **chỉ fit trên train** và học: cột hằng số, cột tương quan > 0,95, median để impute, các category phổ biến (gom hiếm thành `other` rồi One-Hot), signed-log, `StandardScaler`. Val/test chỉ được `transform`.
7. Xử lý mất cân bằng bằng `class_weight` **chỉ trên train**.

**Về NaN:** CIC-IDS2017 chỉ có ~0,05% dòng chứa NaN/inf. Xóa dòng là an toàn và đơn giản hơn impute, đồng thời không tạo giá trị “giả”. Để pipeline vẫn bền vững khi gặp dữ liệu mới (lúc demo/triển khai), bộ tiền xử lý vẫn có bước impute median.

In [ ]:
# ===== 4.1 Hàm làm sạch & lấy mẫu =====
def clean_frame(df, key, name=''):
    log = {'rows_in': int(len(df))}
    drop = [c for c in DS_INFO[key]['drop_cols'] if c in df.columns]
    df = df.drop(columns=drop)                       # (1) bỏ cột định danh / rò rỉ
    log['dropped_id_cols'] = drop
    feats = [c for c in df.columns if c not in LABEL_COLS]
    n_inf = 0
    for c in feats:                                  # (2) inf -> NaN
        if pd.api.types.is_float_dtype(df[c]):
            m = np.isinf(df[c].to_numpy())
            if m.any():
                n_inf += int(m.sum())
                df.loc[m, c] = np.nan
    log['inf_to_nan'] = n_inf
    nan_rows = df[feats].isna().any(axis=1)
    if nan_rows.mean() < 0.01:                       # NaN rất ít -> xóa dòng
        df = df.loc[~nan_rows]
        log['nan_rows_dropped'] = int(nan_rows.sum())
    else:                                            # nhiều NaN -> để NIDSPreprocessor impute median (học trên train)
        log['nan_rows_dropped'] = 0
    before = len(df)
    df = df.drop_duplicates()                        # (3) loại trùng lặp TRƯỚC khi chia tập
    log['duplicates_dropped'] = int(before - len(df))
    df = df.reset_index(drop=True)
    log['rows_out'] = int(len(df))
    print(f'[{key} | {name}]', log)
    return df, log

def stratified_cap_sample(df, label_col, frac, min_keep, seed=SEED):
    # Lấy mẫu phân tầng: mỗi lớp lấy frac, nhưng giữ tối thiểu min_keep mẫu (hoặc toàn bộ nếu lớp nhỏ hơn)
    parts = []
    for _, g in df.groupby(label_col, sort=False):
        k = max(int(round(len(g) * frac)), min(len(g), min_keep))
        parts.append(g.sample(n=k, random_state=seed) if k < len(g) else g)
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def pick_demo_rows(df, n_normal=3, n_attack=6, seed=SEED):
    # Lấy vài bản ghi THÔ của test set (cả Normal và nhiều loại Attack) để demo ở mục 9
    rng = np.random.RandomState(seed)
    normal_ids = np.where(df['y_bin'].to_numpy() == 0)[0]
    idx = list(rng.choice(normal_ids, size=min(n_normal, len(normal_ids)), replace=False))
    attack_classes = df.loc[df['y_bin'] == 1, 'y_multi'].value_counts().index[:n_attack]
    for c in attack_classes:
        idx.append(rng.choice(np.where(df['y_multi'].to_numpy() == c)[0]))
    return df.iloc[idx].reset_index(drop=True)

In [ ]:
# ===== 4.2 Làm sạch, lấy mẫu & chia tập =====
SPLITS, CLEAN_LOG, DEMO_RAW = {}, {}, {}

if 'UNSW-NB15' in ACTIVE:
    key = 'UNSW-NB15'
    tr, log_tr = clean_frame(RAW[key]['train'], key, 'train')
    te, log_te = clean_frame(RAW[key]['test'], key, 'test')
    feats = [c for c in tr.columns if c not in LABEL_COLS]
    overlap = np.isin(rows_hash(tr, feats), rows_hash(te, feats))
    n_ov = int(overlap.sum())
    print(f'[{key}] Loại {n_ov:,} bản ghi train có đặc trưng trùng với test (chống rò rỉ train -> test)')
    tr = tr.loc[~overlap].reset_index(drop=True)
    tr_, va_ = train_test_split(tr, test_size=CONFIG['VAL_SIZE'], stratify=tr['y_multi'], random_state=SEED)
    SPLITS[key] = {'train': tr_.reset_index(drop=True), 'val': va_.reset_index(drop=True), 'test': te}
    CLEAN_LOG[key] = {'train': log_tr, 'test': log_te, 'train_rows_overlap_with_test_removed': n_ov}
    del RAW[key], tr, te, tr_, va_
    gc.collect()

if 'CIC-IDS2017' in ACTIVE:
    key = 'CIC-IDS2017'
    full, log_full = clean_frame(RAW[key]['full'], key, 'full')
    del RAW[key]
    gc.collect()
    vc = full['y_multi'].value_counts()
    rare = vc[vc < 3].index.tolist()
    if rare:
        print('! Bỏ các lớp có < 3 mẫu (không thể chia phân tầng):', rare)
        full = full[~full['y_multi'].isin(rare)].reset_index(drop=True)
    before = full['y_multi'].value_counts()
    frac = CONFIG['CIC_SAMPLE_FRAC']
    if frac < 1.0:
        full = stratified_cap_sample(full, 'y_multi', frac, CONFIG['CIC_MIN_PER_CLASS'])
        gc.collect()
    after = full['y_multi'].value_counts()
    print(f'[{key}] Lấy mẫu phân tầng frac = {frac} (lớp hiếm giữ tối thiểu min(n, CIC_MIN_PER_CLASS) mẫu):')
    display(pd.DataFrame({'Sau làm sạch': before, 'Sau lấy mẫu': after,
                          'Tỉ lệ giữ (%)': (after / before * 100).round(1)}))
    tr, tmp = train_test_split(full, test_size=0.30, stratify=full['y_multi'], random_state=SEED)
    va, te = train_test_split(tmp, test_size=0.50, stratify=tmp['y_multi'], random_state=SEED)
    SPLITS[key] = {'train': tr.reset_index(drop=True), 'val': va.reset_index(drop=True), 'test': te.reset_index(drop=True)}
    CLEAN_LOG[key] = {'full': log_full, 'sample_frac': frac, 'rows_after_sampling': int(len(full)), 'dropped_rare_classes': rare}
    del full, tr, tmp, va, te
    gc.collect()

for key in ACTIVE:
    DEMO_RAW[key] = pick_demo_rows(SPLITS[key]['test'])
    sizes = {s: len(SPLITS[key][s]) for s in ('train', 'val', 'test')}
    print('=' * 100)
    print(f'{key}: kích thước train/val/test = {sizes}')
    dist = pd.DataFrame({s: SPLITS[key][s]['y_multi'].value_counts() for s in ('train', 'val', 'test')}).fillna(0).astype(int)
    display(dist)

**Nhận xét:**
- **UNSW-NB15**: sau khi bỏ `id`, lượng bản ghi trùng lặp bị loại là rất lớn, nhất là ở lớp Generic, Exploits, DoS (các luồng tấn công tự động sinh ra giống hệt nhau). Ngoài ra, thêm một lượng bản ghi train trùng với test cũng bị loại. Hệ quả là kết quả trên test sẽ **thấp hơn** so với nhiều bài báo không loại trùng, nhưng **trung thực hơn**.
- **CIC-IDS2017**: loại ~0,05% dòng NaN/inf và ~10% dòng trùng. Sau lấy mẫu 50%, các lớp lớn (Benign, DoS, DDoS, PortScan) giảm một nửa, còn các lớp nhỏ (Bot, Web Attack, Infiltration, Heartbleed) **được giữ nguyên**. Điều này làm tỉ lệ lớp thay đổi nhẹ so với dữ liệu gốc, nên cần ghi chú khi so sánh với tài liệu khác.
- Bảng phân bố cho thấy val/test giữ **đúng tỉ lệ lớp** như train nhờ chia stratified. Lớp Heartbleed chỉ còn khoảng 1–2 mẫu trong val/test, nên metric của lớp này chỉ mang tính tham khảo.

### 4.3 Bộ tiền xử lý `NIDSPreprocessor` (fit chỉ trên train)
Lớp này đóng gói toàn bộ phép biến đổi đặc trưng để **tái sử dụng** cho cả 2 dataset và cho hàm `predict()` ở mục 9. Bộ tiền xử lý được lưu bằng `joblib` ở mục 10.
- **Chọn cột:** bỏ cột hằng số và cột tương quan > 0,95. Với mỗi cặp tương quan, giữ cột xuất hiện trước và bỏ cột sau (thuật toán tham lam).
- **Số:** `inf → NaN`, impute median của train, rồi **signed-log** `sign(x)·log(1+|x|)` để nén phân phối lệch.
- **Phân loại:** giữ tối đa 20 category có tần suất ≥ 0,5% trong train, còn lại thành `other`, rồi **One-Hot** (category lạ ở test cũng thành `other`).
- **Chuẩn hóa:** `StandardScaler`, quan trọng cho CNN-BiLSTM. Random Forest không cần chuẩn hóa nhưng dùng chung đầu vào để so sánh công bằng.

Chọn **One-Hot** thay vì Label Encoding vì Label Encoding áp đặt thứ tự giả (vd `tcp < udp`) lên mạng nơ-ron. Sau khi gom hiếm, số cột One-Hot vẫn nhỏ (dưới 60).

In [ ]:
class NIDSPreprocessor:
    '''Pipeline tiền xử lý tái sử dụng: chọn cột -> inf/NaN -> signed-log -> One-Hot (gom hiếm) -> StandardScaler.
    Mọi thống kê (cột hằng, cột tương quan, median, category, scaler) CHỈ học trên tập train.'''

    def __init__(self, cat_cols=(), corr_threshold=0.95, min_cat_freq=0.005, max_cats=20,
                 use_log=True, label_cols=('y_bin', 'y_multi')):
        self.cat_cols = list(cat_cols)
        self.corr_threshold = corr_threshold
        self.min_cat_freq = min_cat_freq
        self.max_cats = max_cats
        self.use_log = use_log
        self.label_cols = list(label_cols)

    def fit(self, df):
        X = df.drop(columns=[c for c in self.label_cols if c in df.columns])
        self.cat_cols_ = [c for c in self.cat_cols if c in X.columns]
        num = [c for c in X.columns if c not in self.cat_cols_ and pd.api.types.is_numeric_dtype(X[c])]
        self.ignored_cols_ = [c for c in X.columns if c not in self.cat_cols_ and c not in num]
        # 1) Cột hằng số
        nun = X[num].nunique(dropna=False)
        self.const_cols_ = nun[nun <= 1].index.tolist()
        num = [c for c in num if c not in self.const_cols_]
        # 2) Cột tương quan cao (tính trên mẫu <= 200k dòng của train)
        samp = X[num].sample(min(len(X), 200_000), random_state=SEED).astype('float64')
        corr = samp.replace([np.inf, -np.inf], np.nan).corr().abs().fillna(0).to_numpy()
        iu = np.triu_indices(len(num), k=1)
        mask = corr[iu] > self.corr_threshold
        names = np.array(num, dtype=object)
        self.corr_pairs_ = pd.DataFrame({'feat_a': names[iu[0]][mask], 'feat_b': names[iu[1]][mask],
                                         'abs_corr': corr[iu][mask]}).sort_values('abs_corr', ascending=False)
        drop = set()
        for i, j in zip(iu[0][mask], iu[1][mask]):
            if num[i] not in drop and num[j] not in drop:
                drop.add(num[j])
        self.corr_drop_ = [c for c in num if c in drop]
        self.num_cols_ = [c for c in num if c not in drop]
        # 3) Median để impute
        self.medians_ = X[self.num_cols_].astype('float64').replace([np.inf, -np.inf], np.nan).median().fillna(0)
        # 4) Category phổ biến
        self.cat_levels_ = {}
        for c in self.cat_cols_:
            vc = X[c].astype(str).value_counts(normalize=True)
            self.cat_levels_[c] = vc[vc >= self.min_cat_freq].index[:self.max_cats].tolist()
        # 5) Scaler
        Z, self.feature_names_ = self._build(X)
        self.scaler_ = StandardScaler().fit(Z)
        return self

    def _build(self, X):
        num = X.reindex(columns=self.num_cols_)          # cột thiếu -> NaN -> median
        if (num.dtypes == object).any():
            num = num.apply(pd.to_numeric, errors='coerce')
        arr = num.to_numpy(dtype=np.float32, copy=True)
        arr[~np.isfinite(arr)] = np.nan
        rows, cols = np.where(np.isnan(arr))
        if len(rows):
            arr[rows, cols] = self.medians_.to_numpy(dtype=np.float32)[cols]
        if self.use_log:
            arr = np.sign(arr) * np.log1p(np.abs(arr))
        parts, names = [arr.astype(np.float32)], list(self.num_cols_)
        for c in self.cat_cols_:
            keep = self.cat_levels_[c]
            levels = np.array(keep + ['other'], dtype=str)
            col = X[c].astype(str).to_numpy() if c in X.columns else np.full(len(X), 'other')
            col = np.where(np.isin(col, keep), col, 'other').astype(str)
            parts.append((col[:, None] == levels[None, :]).astype(np.float32))
            names += [f'{c}={l}' for l in levels]
        return np.hstack(parts).astype(np.float32), names

    def transform(self, df):
        Z, _ = self._build(df)
        return self.scaler_.transform(Z).astype(np.float32)

In [ ]:
# ===== 4.4 Fit bộ tiền xử lý trên train, transform train/val/test, mã hóa nhãn =====
DATA = {}
thr = CONFIG['CORR_THRESHOLD']
for key in ACTIVE:
    sp = SPLITS[key]
    t0 = time.time()
    prep = NIDSPreprocessor(cat_cols=DS_INFO[key]['cat_cols'], corr_threshold=thr,
                            min_cat_freq=CONFIG['MIN_CAT_FREQ'], max_cats=CONFIG['MAX_CATS']).fit(sp['train'])
    X = {s: prep.transform(sp[s]) for s in ('train', 'val', 'test')}
    le = LabelEncoder().fit(sp['train']['y_multi'].astype(str))
    y = {'binary': tuple(sp[s]['y_bin'].to_numpy().astype(np.int32) for s in ('train', 'val', 'test')),
         'multi': tuple(le.transform(sp[s]['y_multi'].astype(str)).astype(np.int32) for s in ('train', 'val', 'test'))}
    classes_multi = [str(c) for c in le.classes_]
    DATA[key] = {'X_train': X['train'], 'X_val': X['val'], 'X_test': X['test'], 'y': y,
                 'prep': prep, 'le': le, 'feature_names': prep.feature_names_,
                 'classes': {'binary': ['Normal', 'Attack'], 'multi': classes_multi},
                 'normal_idx': {'binary': 0, 'multi': classes_multi.index(DS_INFO[key]['normal'])}}
    shapes = {s: X[s].shape for s in X}
    n_onehot = {c: len(v) + 1 for c, v in prep.cat_levels_.items()}
    print('=' * 100)
    print(f'{key}: tiền xử lý xong trong {time.time() - t0:.1f}s | shapes = {shapes}')
    print(f'  - Cột hằng số bị loại ({len(prep.const_cols_)}): {prep.const_cols_}')
    print(f'  - Cột tương quan > {thr} bị loại ({len(prep.corr_drop_)}): {prep.corr_drop_}')
    print(f'  - Cột phân loại -> số cột One-Hot: {n_onehot}')
    print(f'  - Số đặc trưng cuối cùng: {len(prep.feature_names_)}')
    print(f'  - Lớp đa lớp (LabelEncoder): {dict(enumerate(classes_multi))}')
    display(prep.corr_pairs_.head(15).round(4))

del SPLITS, X
gc.collect();

**Nhận xét:**
- **CIC-IDS2017**: khoảng 8 cột hằng số và hơn 20 cột tương quan cao bị loại, nên số đặc trưng giảm từ ~78 xuống còn khoảng 45–50. Mô hình nhẹ hơn và nhanh hơn mà gần như không mất thông tin.
- **UNSW-NB15**: khoảng 39 cột số ban đầu, sau khi loại vài cột tương quan cao và cộng thêm One-Hot của `proto`/`service`/`state` (đã gom hiếm), tổng số đặc trưng vào khoảng 50.
- Mọi thống kê đều chỉ học từ train, nên val/test hoàn toàn “chưa được nhìn thấy”. Đây là điều kiện để đánh giá trung thực.

### 4.5 Kiểm tra rò rỉ dữ liệu (sanity check)
Kiểm tra sau tiền xử lý: số vector đặc trưng test/val trùng hệt với train, còn NaN/inf hay không, và còn sót cột định danh hay không.

In [ ]:
def matrix_overlap(A, B):
    ha = pd.util.hash_pandas_object(pd.DataFrame(A), index=False).to_numpy()
    hb = pd.util.hash_pandas_object(pd.DataFrame(B), index=False).to_numpy()
    return int(np.isin(hb, ha).sum())

leak_rows = []
for key in ACTIVE:
    d = DATA[key]
    n_test_dup = matrix_overlap(d['X_train'], d['X_test'])
    leak_rows.append({'Dataset': key, 'Train': len(d['X_train']), 'Val': len(d['X_val']), 'Test': len(d['X_test']),
                      'Số đặc trưng': d['X_train'].shape[1],
                      'Test trùng Train': n_test_dup,
                      '% Test trùng': round(n_test_dup / len(d['X_test']) * 100, 3),
                      'Val trùng Train': matrix_overlap(d['X_train'], d['X_val']),
                      'Còn NaN/inf': bool(not (np.isfinite(d['X_train']).all() and np.isfinite(d['X_test']).all())),
                      'Cột định danh còn sót': [c for c in DS_INFO[key]['drop_cols'] if c in d['feature_names']]})
display(pd.DataFrame(leak_rows))

**Nhận xét:**
- Không còn NaN/inf và không còn cột định danh trong ma trận đặc trưng.
- Số vector test trùng train phải **bằng 0 hoặc rất nhỏ**. Phần trùng nhỏ còn lại (nếu có) là do sau khi bỏ cột tương quan và gom category hiếm thành `other`, hai bản ghi vốn khác nhau ở dữ liệu thô trở nên giống hệt nhau. Đây không phải rò rỉ thật.
- Nếu sau này kết quả gần 100% (thường gặp ở CIC-IDS2017), bảng này cho thấy nguyên nhân **không phải** do trùng lặp hay cột ID. Lý do thật là tính dễ phân tách của dữ liệu mô phỏng (xem phân tích ở mục 7–8).

### 4.6 Xử lý mất cân bằng lớp
- **Random Forest:** `class_weight='balanced'` với trọng số `w_c = N / (K · N_c)`, đúng như yêu cầu.
- **CNN-BiLSTM:** dùng **sqrt-balanced** `w_c = sqrt(N / (K · N_c))`, cắt ở tối đa 50. Với CIC-IDS2017, trọng số balanced của Heartbleed lên tới hàng chục nghìn, làm gradient bùng nổ và huấn luyện mất ổn định. Lấy căn bậc hai vẫn ưu tiên lớp hiếm nhưng giữ huấn luyện ổn định.
- **Không dùng SMOTE** trong cấu hình mặc định: (i) dữ liệu luồng mạng có nhiều đặc trưng rời rạc/đếm, nên nội suy SMOTE dễ sinh mẫu phi thực tế; (ii) với ~900k mẫu train, SMOTE rất tốn RAM/thời gian; (iii) class_weight đạt hiệu quả tương đương trong nhiều nghiên cứu IDS. Nếu muốn thử, chỉ được áp dụng `imblearn.over_sampling.SMOTE` trên `X_train`, tuyệt đối không áp dụng cho val/test.

In [ ]:
def class_weights(y, mode='sqrt', max_w=50.0):
    classes = np.unique(y)
    w = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    if mode == 'sqrt':
        w = np.sqrt(w)
    w = np.minimum(w, max_w)
    return {int(c): float(v) for c, v in zip(classes, w)}

for key in ACTIVE:
    for task in TASKS:
        ytr = DATA[key]['y'][task][0]
        names = DATA[key]['classes'][task]
        cnt = np.bincount(ytr, minlength=len(names))
        wb = class_weights(ytr, 'balanced', np.inf)
        ws = class_weights(ytr, 'sqrt')
        tbl = pd.DataFrame({'Số mẫu train': cnt,
                            'balanced (RF)': [wb.get(i, np.nan) for i in range(len(names))],
                            'sqrt-balanced (DL)': [ws.get(i, np.nan) for i in range(len(names))]}, index=names)
        print(f'{key} – {TASK_VN[task]}: trọng số lớp')
        display(tbl.round(3))

**Nhận xét:** với bài toán nhị phân, trọng số hai lớp chênh lệch không nhiều. Với bài toán đa lớp CIC-IDS2017, trọng số balanced của Heartbleed/Infiltration lớn hơn Benign **hàng chục nghìn lần**, còn sqrt-balanced thu hẹp khoảng cách này về mức vài chục lần. Đây là lựa chọn hợp lý cho huấn luyện bằng gradient.

## 5. Mô hình 1 – Baseline: Random Forest

**Làm gì:** cho mỗi dataset × mỗi bài toán:
1. Train **baseline** với tham số hợp lý: `n_estimators=200`, `max_features='sqrt'`, `min_samples_leaf=2` (giới hạn kích thước cây), `class_weight='balanced'`, `n_jobs=-1`.
2. **RandomizedSearchCV** (`scoring='f1_macro'`, 3-fold stratified) trên **subset phân tầng** `RF_TUNE_SUBSET` mẫu của train, với 100 cây để tìm nhanh. Sau đó train lại cấu hình tốt nhất trên **toàn bộ train** với 200 cây.
3. Chọn biến thể (baseline hoặc tuned) có **F1-macro trên validation** cao hơn. Tập test **không** được dùng để chọn mô hình.
4. Ghi lại thời gian train, thời gian dự đoán / 1000 mẫu, và kích thước mô hình (MB, đo bằng pickle).

**Tại sao Random Forest làm baseline:** mạnh với dữ liệu bảng, không cần chuẩn hóa, ít nhạy với outlier, cho feature importance, và là baseline phổ biến nhất trong các nghiên cứu NIDS.

In [ ]:
# ===== 5.1 Hàm huấn luyện Random Forest =====
RESULTS = {}     # (dataset, task, model) -> dict kết quả trên test
RF_MODELS = {}   # (dataset, task) -> mô hình RF được chọn
RF_SEARCH = {}   # (dataset, task) -> bảng kết quả RandomizedSearchCV

RF_BASE_PARAMS = dict(n_estimators=200, max_features='sqrt', min_samples_leaf=2,
                      class_weight='balanced', n_jobs=-1, random_state=SEED)
RF_PARAM_DIST = {
    'max_depth': [None, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.3],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced', 'balanced_subsample'],
}

def model_size_mb(obj):
    return len(pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)) / 1024 ** 2

def timed_predict(predict_fn, X):
    # Trả về (xác suất, thời gian dự đoán tính bằng ms / 1000 mẫu)
    t0 = time.perf_counter()
    P = predict_fn(X)
    dt = time.perf_counter() - t0
    return P, dt / len(X) * 1000 * 1000

def subsample_idx(y, n_target, min_per_class=30, seed=SEED):
    # Subset phân tầng, đảm bảo mỗi lớp có ít nhất min_per_class mẫu (nếu đủ)
    if len(y) <= n_target:
        return np.arange(len(y))
    rng = np.random.RandomState(seed)
    frac = n_target / len(y)
    idx = []
    for c in np.unique(y):
        ids = np.where(y == c)[0]
        k = min(len(ids), max(int(round(len(ids) * frac)), min_per_class))
        idx.append(rng.choice(ids, size=k, replace=False))
    return np.sort(np.concatenate(idx))

def run_random_forest(key, task):
    d = DATA[key]
    ytr, yva, yte = d['y'][task]
    Xtr, Xva, Xte = d['X_train'], d['X_val'], d['X_test']
    print('=' * 100)
    print(f'[RF] {key} – {TASK_VN[task]} | X_train = {Xtr.shape}')
    t0 = time.time()
    base = RandomForestClassifier(**RF_BASE_PARAMS).fit(Xtr, ytr)
    t_base = time.time() - t0
    f1_base = f1_score(yva, base.predict(Xva), average='macro')
    print(f'  Baseline : train {t_base:.1f}s | F1-macro (val) = {f1_base:.4f}')
    cands = [('baseline', base, t_base, f1_base, dict(RF_BASE_PARAMS))]
    if CONFIG['RF_TUNE']:
        idx = subsample_idx(ytr, CONFIG['RF_TUNE_SUBSET'])
        search = RandomizedSearchCV(
            RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED),
            param_distributions=RF_PARAM_DIST, n_iter=CONFIG['RF_TUNE_ITER'], scoring='f1_macro',
            cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
            refit=False, random_state=SEED, n_jobs=1, verbose=1)
        t0 = time.time()
        search.fit(Xtr[idx], ytr[idx])
        t_search = time.time() - t0
        best_params = {**RF_BASE_PARAMS, **search.best_params_}
        print(f'  RandomizedSearchCV trên {len(idx):,} mẫu: {t_search:.1f}s | best CV F1-macro = {search.best_score_:.4f}')
        print(f'  Best params: {search.best_params_}')
        t0 = time.time()
        tuned = RandomForestClassifier(**best_params).fit(Xtr, ytr)
        t_tuned = time.time() - t0
        f1_tuned = f1_score(yva, tuned.predict(Xva), average='macro')
        print(f'  Tuned    : train {t_tuned:.1f}s | F1-macro (val) = {f1_tuned:.4f}')
        cands.append(('tuned', tuned, t_tuned, f1_tuned, best_params))
        cvr = pd.DataFrame(search.cv_results_)[['params', 'mean_test_score', 'std_test_score', 'mean_fit_time', 'rank_test_score']]
        RF_SEARCH[(key, task)] = cvr.sort_values('rank_test_score').reset_index(drop=True)
    variant, model, t_train, f1v, params = max(cands, key=lambda c: c[3])
    print(f'  -> Chọn biến thể: {variant}')
    P, inf_ms = timed_predict(model.predict_proba, Xte)
    RF_MODELS[(key, task)] = model
    RESULTS[(key, task, 'RandomForest')] = {
        'y_pred': P.argmax(axis=1), 'y_proba': P, 'train_time': t_train, 'infer_ms': inf_ms,
        'size_mb': model_size_mb(model), 'variant': variant, 'params': params, 'val_f1': f1v}
    rows = [{'Dataset': key, 'Task': task, 'Biến thể': v, 'F1-macro (val)': round(f, 4),
             'Train time (s)': round(t, 1), 'Được chọn': v == variant} for v, _, t, f, _ in cands]
    del cands, base
    gc.collect()
    return rows

In [ ]:
# ===== 5.2 Huấn luyện RF cho mọi (dataset, task) =====
rf_rows = []
for key in ACTIVE:
    for task in TASKS:
        rf_rows += run_random_forest(key, task)

RF_SUMMARY = pd.DataFrame(rf_rows)
display(RF_SUMMARY)
rf_info = pd.DataFrame([{'Dataset': k, 'Task': t, 'Biến thể': r['variant'],
                         'Train time (s)': round(r['train_time'], 1),
                         'Inference (ms/1000 mẫu)': round(r['infer_ms'], 3),
                         'Kích thước (MB)': round(r['size_mb'], 2)}
                        for (k, t, m), r in RESULTS.items() if m == 'RandomForest'])
display(rf_info)

**Nhận xét:**
- RandomizedSearchCV chỉ chạy trên subset, nên tham số tốt nhất **không đảm bảo** tốt hơn baseline khi train trên toàn bộ dữ liệu. Vì vậy việc chọn bằng validation là cần thiết. Thường thì chênh lệch F1-macro giữa hai biến thể khá nhỏ (< 1 điểm %), cho thấy RF **ít nhạy với siêu tham số**.
- Thời gian train RF tăng gần tuyến tính theo số mẫu. CIC-IDS2017 (~900k mẫu train) lâu hơn UNSW-NB15 nhiều lần.
- **Kích thước mô hình** RF khá lớn (hàng chục đến hàng trăm MB) vì cây không giới hạn độ sâu. Đây là điểm cần cân nhắc khi triển khai trên thiết bị biên. Có thể giảm bằng `max_depth` hoặc tăng `min_samples_leaf`.
- Thời gian dự đoán chỉ khoảng vài ms đến vài chục ms / 1000 mẫu trên CPU, đủ đáp ứng realtime cho hàng chục nghìn luồng/giây.

In [ ]:
# ===== 5.3 Feature importance – top 20 =====
fig, axes = plt.subplots(len(ACTIVE), 2, figsize=(22, 7.5 * len(ACTIVE)), squeeze=False)
for i, key in enumerate(ACTIVE):
    for j, task in enumerate(TASKS):
        ax = axes[i, j]
        if (key, task) not in RF_MODELS:
            ax.axis('off'); continue
        imp = pd.Series(RF_MODELS[(key, task)].feature_importances_, index=DATA[key]['feature_names'])
        imp = imp.sort_values(ascending=False).head(20)
        sns.barplot(x=imp.values, y=imp.index, color='#4C72B0', ax=ax)
        ax.set_title(f'Random Forest – Top 20 feature importance – {key} – {TASK_VN[task]}')
        ax.set_xlabel('Mean decrease in impurity (Gini importance)'); ax.set_ylabel('Đặc trưng')
plt.tight_layout(); save_fig('rf_feature_importance_top20'); plt.show()

for (key, task), cvr in RF_SEARCH.items():
    print(f'Top 5 cấu hình RandomizedSearchCV – {key} – {TASK_VN[task]}')
    display(cvr.head(5))

**Nhận xét:**
- **UNSW-NB15**: các đặc trưng liên quan TTL (`sttl`, `dttl`, `ct_state_ttl`) và kích thước gói/byte (`sbytes`, `smean`, `dmean`, `dbytes`) thường đứng đầu. Điều này khớp với EDA, nơi TTL phân tách Normal và Attack rất rõ. Ở bài toán đa lớp, các đặc trưng đếm kết nối (`ct_srv_src`, `ct_dst_src_ltm`…) và `service=dns` cũng quan trọng hơn, vì chúng giúp phân biệt Generic và Reconnaissance.
- **CIC-IDS2017**: nhóm kích thước gói (`Packet Length`, `Bwd Packet Length Max/Mean`, `Avg Bwd Segment Size`), cửa sổ TCP ban đầu (`Init_Win_bytes_forward/backward`) và `Destination Port` chiếm ưu thế. `Init_Win_bytes` phụ thuộc mạnh vào hệ điều hành/công cụ tấn công trong testbed, nên là dấu hiệu mô hình có thể học **đặc thù môi trường** thay vì hành vi tấn công tổng quát.
- Lưu ý: Gini importance thiên vị đặc trưng có nhiều giá trị khác nhau. Mục 7.6 dùng SHAP để kiểm chứng.

## 6. Mô hình 2 – Deep Learning: 1D-CNN + BiLSTM

**Ý tưởng:** xem mỗi bản ghi (vector đặc trưng đã chuẩn hóa) như một “chuỗi” 1 chiều có độ dài `n_features`, nên reshape thành `(samples, n_features, 1)`.
- **Conv1D** học các tổ hợp cục bộ giữa các đặc trưng lân cận (vd nhóm bytes/packets/rate).
- **MaxPooling** giảm chiều dài chuỗi.
- **BiLSTM** tổng hợp thông tin trên toàn bộ chuỗi theo cả 2 chiều.
- **Dense + Dropout** thực hiện phân loại.

Lưu ý: dữ liệu bảng **không có thứ tự tự nhiên** giữa các đặc trưng, nên kiến trúc này chủ yếu khai thác khả năng học biểu diễn phi tuyến. Đây là kiến trúc phổ biến trong tài liệu IDS, và việc so sánh nó với RF chính là mục tiêu của project.

**Kiến trúc:** `Conv1D(64, k=3) → BatchNorm → MaxPool(2) → Conv1D(128, k=3) → BatchNorm → MaxPool(2) → Bidirectional(LSTM(64)) → Dense(128, ReLU) → Dropout(0.3) → Output` (sigmoid cho nhị phân, softmax cho đa lớp).

**Huấn luyện:**
- Adam(lr=1e-3), loss `binary_crossentropy` / `sparse_categorical_crossentropy`, class_weight sqrt-balanced.
- batch 1024, tối đa 30 epoch.
- Callbacks: `EarlyStopping(patience=5, restore_best_weights=True)`, `ReduceLROnPlateau(factor=0.5, patience=2)`, `ModelCheckpoint` (lưu mô hình tốt nhất theo `val_loss`).

In [ ]:
# ===== 6.1 Xây dựng mô hình =====
DL_MODELS, HISTORIES = {}, {}

def build_cnn_bilstm(n_features, n_classes, name='cnn_bilstm'):
    inp = keras.Input(shape=(n_features, 1), name='flow_features')
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2, padding='same')(x)
    x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2, padding='same')(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    if n_classes == 2:
        out = layers.Dense(1, activation='sigmoid', name='attack_prob')(x)
        loss = 'binary_crossentropy'
    else:
        out = layers.Dense(n_classes, activation='softmax', name='class_prob')(x)
        loss = 'sparse_categorical_crossentropy'
    model = keras.Model(inp, out, name=name)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss=loss, metrics=['accuracy'])
    return model

def to_proba(p):
    # Chuẩn hóa đầu ra Keras thành ma trận xác suất (n, n_classes)
    p = np.asarray(p, dtype=np.float64)
    if p.ndim == 1 or p.shape[1] == 1:
        p = p.reshape(-1)
        return np.column_stack([1.0 - p, p])
    return p / p.sum(axis=1, keepdims=True)

_demo = build_cnn_bilstm(DATA[ACTIVE[0]]['X_train'].shape[1], 2, name='cnn_bilstm_demo')
_demo.summary()
for key in ACTIVE:
    for task in TASKS:
        _m = build_cnn_bilstm(DATA[key]['X_train'].shape[1], len(DATA[key]['classes'][task]))
        print(f'{key} – {TASK_VN[task]}: input = ({DATA[key]["X_train"].shape[1]}, 1) | {_m.count_params():,} tham số')
del _demo, _m
keras.backend.clear_session()
gc.collect();

**Nhận xét:** mô hình chỉ có khoảng **150 nghìn tham số**, chiếm vài trăm KB đến khoảng 2 MB khi lưu, nhỏ hơn rất nhiều so với Random Forest (hàng chục đến hàng trăm MB). Sau 2 lớp MaxPool, chuỗi dài ~50 giảm còn ~13 bước, đủ ngắn để BiLSTM (cuDNN) chạy nhanh trên GPU.

In [ ]:
# ===== 6.2 Huấn luyện CNN-BiLSTM cho mọi (dataset, task) =====
def run_dl(key, task):
    d = DATA[key]
    ytr, yva, yte = d['y'][task]
    n_classes = len(d['classes'][task])
    n_feat = d['X_train'].shape[1]
    keras.backend.clear_session()
    set_seed()
    model = build_cnn_bilstm(n_feat, n_classes, name=f'cnn_bilstm_{slug(key)}_{task}')
    ckpt = os.path.join(MODEL_DIR, f'dl_{slug(key)}_{task}_best.keras')
    cbs = [callbacks.EarlyStopping(monitor='val_loss', patience=CONFIG['DL_PATIENCE'], restore_best_weights=True, verbose=1),
           callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
           callbacks.ModelCheckpoint(ckpt, monitor='val_loss', save_best_only=True, verbose=0)]
    cw = class_weights(ytr, 'sqrt')
    print('=' * 100)
    print(f'[DL] {key} – {TASK_VN[task]} | X_train = {d["X_train"].shape} | class_weight = { {k: round(v, 2) for k, v in cw.items()} }')
    t0 = time.time()
    hist = model.fit(d['X_train'][..., np.newaxis], ytr,
                     validation_data=(d['X_val'][..., np.newaxis], yva),
                     epochs=CONFIG['DL_EPOCHS'], batch_size=CONFIG['DL_BATCH'],
                     class_weight=cw, callbacks=cbs, verbose=2)
    t_train = time.time() - t0
    predict_fn = lambda X: to_proba(model.predict(X[..., np.newaxis], batch_size=4096, verbose=0))
    P, inf_ms = timed_predict(predict_fn, d['X_test'])
    final_path = os.path.join(MODEL_DIR, f'dl_{slug(key)}_{task}.keras')
    model.save(final_path)
    size_mb = os.path.getsize(final_path) / 1024 ** 2
    n_ep = len(hist.history['loss'])
    print(f'  -> {n_ep} epoch, train {t_train:.1f}s, inference {inf_ms:.3f} ms/1000 mẫu, size {size_mb:.2f} MB')
    DL_MODELS[(key, task)] = model
    HISTORIES[(key, task)] = {k: [float(v) for v in vals] for k, vals in hist.history.items()}
    RESULTS[(key, task, 'CNN-BiLSTM')] = {
        'y_pred': P.argmax(axis=1), 'y_proba': P, 'train_time': t_train, 'infer_ms': inf_ms,
        'size_mb': size_mb, 'epochs': n_ep, 'n_params': int(model.count_params())}
    gc.collect()

for key in ACTIVE:
    for task in TASKS:
        run_dl(key, task)

dl_info = pd.DataFrame([{'Dataset': k, 'Task': t, 'Epochs': r['epochs'], 'Tham số': r['n_params'],
                         'Train time (s)': round(r['train_time'], 1),
                         'Inference (ms/1000 mẫu)': round(r['infer_ms'], 3),
                         'Kích thước (MB)': round(r['size_mb'], 2)}
                        for (k, t, m), r in RESULTS.items() if m == 'CNN-BiLSTM'])
display(dl_info)

In [ ]:
# ===== 6.3 Learning curves =====
items = list(HISTORIES.items())
fig, axes = plt.subplots(len(items), 2, figsize=(16, 4.2 * len(items)), squeeze=False)
for i, ((key, task), h) in enumerate(items):
    ep = np.arange(1, len(h['loss']) + 1)
    best = int(np.argmin(h['val_loss'])) + 1
    for j, metric in enumerate(['loss', 'accuracy']):
        ax = axes[i, j]
        if metric not in h or ('val_' + metric) not in h:
            ax.axis('off'); continue
        ax.plot(ep, h[metric], 'o-', ms=3, label='Train')
        ax.plot(ep, h['val_' + metric], 's-', ms=3, label='Validation')
        ax.axvline(best, color='gray', ls='--', lw=1, label=f'Best epoch (val_loss) = {best}')
        ax.set_title(f'CNN-BiLSTM – {key} – {TASK_VN[task]} – {metric}')
        ax.set_xlabel('Epoch'); ax.set_ylabel(metric.capitalize())
        ax.legend()
plt.tight_layout(); save_fig('dl_learning_curves'); plt.show()

**Nhận xét – overfitting/underfitting:**
- Loss train giảm nhanh trong vài epoch đầu rồi đi ngang. Loss validation bám sát train và **không tăng trở lại rõ rệt**, nên không có dấu hiệu overfitting nghiêm trọng, nhờ BatchNorm, Dropout, EarlyStopping và dữ liệu lớn. EarlyStopping thường dừng trước epoch 30 và khôi phục trọng số tốt nhất (đường đứt nét).
- **Accuracy train có thể thấp hơn validation** ở một số epoch. Đây không phải lỗi: (i) Dropout chỉ hoạt động lúc train; (ii) loss/accuracy train được tính trong khi trọng số còn thay đổi; (iii) class_weight làm mô hình chấp nhận sai thêm vài mẫu lớp lớn để đoán đúng lớp hiếm.
- **UNSW-NB15 đa lớp** có val loss cao hơn hẳn và dao động nhiều hơn. Điều này phản ánh giới hạn của dữ liệu (các lớp DoS/Exploits/Analysis/Backdoor chồng lấn nhau) hơn là underfitting của mô hình: tăng độ phức tạp mô hình thường không cải thiện thêm.
- ReduceLROnPlateau giảm learning rate khi val loss chững lại, giúp loss giảm thêm một chút ở các epoch cuối.

## 7. Đánh giá kết quả trên TEST set

Cả hai mô hình được đánh giá trên **cùng một tập test** (cùng số mẫu, cùng thứ tự) cho mỗi dataset × bài toán, nên so sánh là công bằng.

**Các chỉ số:**
- Accuracy, Precision, Recall, F1 (macro và weighted), cùng `classification_report` đầy đủ.
- **Chỉ số riêng cho IDS** (với bài toán đa lớp, quy về Attack = mọi lớp khác lớp bình thường):
  - **Detection Rate (DR)** = TP / (TP + FN) = Recall của lớp Attack, tức tỉ lệ tấn công bị phát hiện.
  - **False Alarm Rate (FAR)** = FP / (FP + TN) = FPR, tức tỉ lệ lưu lượng bình thường bị báo động nhầm.
  - **Missed Attack Rate (FNR)** = FN / (TP + FN) = 1 − DR.
- **ROC-AUC** và **PR-AUC** (Average Precision). Đa lớp dùng one-vs-rest, trung bình macro. PR-AUC có ý nghĩa hơn ROC-AUC khi dữ liệu mất cân bằng.
- **Confusion matrix** dạng số lượng và dạng chuẩn hóa theo hàng (mỗi hàng là recall của lớp thật).

In [ ]:
# ===== 7.0 Hàm đánh giá =====
METRICS, PER_CLASS_F1, CMS = {}, {}, {}

def ids_metrics(y_true, y_pred, normal_idx):
    t = y_true != normal_idx
    p = y_pred != normal_idx
    tp, fn = np.sum(t & p), np.sum(t & ~p)
    fp, tn = np.sum(~t & p), np.sum(~t & ~p)
    dr = tp / max(tp + fn, 1)
    far = fp / max(fp + tn, 1)
    fnr = fn / max(tp + fn, 1)
    return float(dr), float(far), float(fnr)

def compute_metrics(key, task, model_name):
    r, d = RESULTS[(key, task, model_name)], DATA[key]
    y, yp, P = d['y'][task][2], r['y_pred'], r['y_proba']
    classes = d['classes'][task]
    labels = list(range(len(classes)))
    m = {'Model': model_name, 'Dataset': key, 'Task': task,
         'Accuracy': accuracy_score(y, yp),
         'Precision_macro': precision_score(y, yp, average='macro', labels=labels, zero_division=0),
         'Recall_macro': recall_score(y, yp, average='macro', labels=labels, zero_division=0),
         'F1_macro': f1_score(y, yp, average='macro', labels=labels, zero_division=0),
         'Precision_weighted': precision_score(y, yp, average='weighted', labels=labels, zero_division=0),
         'Recall_weighted': recall_score(y, yp, average='weighted', labels=labels, zero_division=0),
         'F1_weighted': f1_score(y, yp, average='weighted', labels=labels, zero_division=0)}
    m['DR'], m['FAR'], m['FNR'] = ids_metrics(y, yp, d['normal_idx'][task])
    try:
        if len(classes) == 2:
            m['ROC_AUC'] = roc_auc_score(y, P[:, 1])
            m['PR_AUC'] = average_precision_score(y, P[:, 1])
        else:
            m['ROC_AUC'] = roc_auc_score(y, P, multi_class='ovr', average='macro', labels=labels)
            m['PR_AUC'] = average_precision_score(label_binarize(y, classes=labels), P, average='macro')
    except Exception as e:
        print('  ! Không tính được AUC:', e)
        m['ROC_AUC'], m['PR_AUC'] = np.nan, np.nan
    m.update({'Train_time_s': r['train_time'], 'Infer_ms_per_1000': r['infer_ms'], 'Size_MB': r['size_mb']})
    METRICS[(key, task, model_name)] = m
    PER_CLASS_F1[(key, task, model_name)] = pd.Series(
        f1_score(y, yp, average=None, labels=labels, zero_division=0), index=classes)
    CMS[(key, task, model_name)] = confusion_matrix(y, yp, labels=labels)
    return m

def plot_confusion(key, task, model_name):
    cm = CMS[(key, task, model_name)]
    classes = DATA[key]['classes'][task]
    cmn = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    size = max(5.5, 0.75 * len(classes) + 2.5)
    fig, axes = plt.subplots(1, 2, figsize=(2 * size + 3, size))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes,
                ax=axes[0], cbar=False, annot_kws={'fontsize': 8})
    sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues', xticklabels=classes, yticklabels=classes,
                ax=axes[1], vmin=0, vmax=1, annot_kws={'fontsize': 8}, cbar_kws={'label': 'Tỉ lệ theo hàng'})
    axes[0].set_title(f'{model_name} – {key} – {TASK_VN[task]}: Confusion matrix (số lượng)')
    axes[1].set_title(f'{model_name} – {key} – {TASK_VN[task]}: Confusion matrix (chuẩn hóa theo hàng)')
    for ax in axes:
        ax.set_xlabel('Nhãn dự đoán'); ax.set_ylabel('Nhãn thật')
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout(); save_fig(f'cm_{model_name}_{key}_{task}'); plt.show()

def plot_roc_pr(key, task):
    d = DATA[key]
    y = d['y'][task][2]
    classes = d['classes'][task]
    n = len(classes)
    models = [m for m in MODEL_NAMES if (key, task, m) in RESULTS]
    if n == 2:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
        for m in models:
            p = RESULTS[(key, task, m)]['y_proba'][:, 1]
            fpr, tpr, _ = roc_curve(y, p)
            prec, rec, _ = precision_recall_curve(y, p)
            axes[0].plot(fpr, tpr, lw=2, label=f'{m} (ROC-AUC = {roc_auc_score(y, p):.4f})')
            axes[1].plot(rec, prec, lw=2, label=f'{m} (PR-AUC = {average_precision_score(y, p):.4f})')
        axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Ngẫu nhiên')
        axes[1].axhline(y.mean(), color='k', ls='--', lw=1, label=f'Ngẫu nhiên (tỉ lệ Attack = {y.mean():.2f})')
        axes[0].set_title(f'ROC curve – {key} – {TASK_VN[task]}')
        axes[0].set_xlabel('False Positive Rate (FAR)'); axes[0].set_ylabel('True Positive Rate (DR)')
        axes[1].set_title(f'Precision-Recall curve – {key} – {TASK_VN[task]}')
        axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
        axes[0].legend(loc='lower right'); axes[1].legend(loc='lower left')
    else:
        Yb = label_binarize(y, classes=list(range(n)))
        cmap = plt.get_cmap('tab10')
        fig, axes = plt.subplots(len(models), 2, figsize=(17, 6 * len(models)), squeeze=False)
        for i, m in enumerate(models):
            P = RESULTS[(key, task, m)]['y_proba']
            for c in range(n):
                if Yb[:, c].sum() == 0:
                    continue
                fpr, tpr, _ = roc_curve(Yb[:, c], P[:, c])
                prec, rec, _ = precision_recall_curve(Yb[:, c], P[:, c])
                ap = average_precision_score(Yb[:, c], P[:, c])
                axes[i, 0].plot(fpr, tpr, color=cmap(c % 10), lw=1.5, label=f'{classes[c]} (AUC = {auc(fpr, tpr):.3f})')
                axes[i, 1].plot(rec, prec, color=cmap(c % 10), lw=1.5, label=f'{classes[c]} (AP = {ap:.3f})')
            axes[i, 0].plot([0, 1], [0, 1], 'k--', lw=1)
            axes[i, 0].set_title(f'{m} – ROC one-vs-rest – {key}')
            axes[i, 0].set_xlabel('False Positive Rate'); axes[i, 0].set_ylabel('True Positive Rate')
            axes[i, 1].set_title(f'{m} – Precision-Recall one-vs-rest – {key}')
            axes[i, 1].set_xlabel('Recall'); axes[i, 1].set_ylabel('Precision')
            axes[i, 0].legend(loc='lower right', fontsize=8); axes[i, 1].legend(loc='lower left', fontsize=8)
    plt.tight_layout(); save_fig(f'roc_pr_{key}_{task}'); plt.show()

def evaluate_block(key, task):
    if key not in ACTIVE:
        print(f'Bỏ qua {key} (dataset không được chạy).')
        return
    d = DATA[key]
    y = d['y'][task][2]
    classes = d['classes'][task]
    for m in MODEL_NAMES:
        if (key, task, m) not in RESULTS:
            continue
        res = compute_metrics(key, task, m)
        print('=' * 100)
        print(f'{m} – {key} – {TASK_VN[task]}  (test = {len(y):,} mẫu)')
        print(classification_report(y, RESULTS[(key, task, m)]['y_pred'], labels=list(range(len(classes))),
                                    target_names=classes, digits=4, zero_division=0))
        print(f'Detection Rate = {res["DR"]:.4f} | False Alarm Rate = {res["FAR"]:.4f} | Missed Attack Rate = {res["FNR"]:.4f}')
        print(f'ROC-AUC = {res["ROC_AUC"]:.4f} | PR-AUC = {res["PR_AUC"]:.4f}')
        plot_confusion(key, task, m)
    plot_roc_pr(key, task)
    tbl = pd.DataFrame([METRICS[(key, task, m)] for m in MODEL_NAMES if (key, task, m) in METRICS])
    display(tbl.drop(columns=['Dataset', 'Task']).set_index('Model').round(4))

### 7.1 UNSW-NB15 – Phân loại nhị phân

In [ ]:
evaluate_block('UNSW-NB15', 'binary')

**Nhận xét:**
- Với UNSW-NB15 nhị phân, cả hai mô hình thường đạt Accuracy/F1 khoảng **0,87–0,92** và ROC-AUC > 0,97. Kết quả thấp hơn các con số “99%” trong một số bài báo, vì ta dùng đúng split gốc **và đã loại trùng lặp/giao train–test**.
- **DR cao (~0,95–0,99)** nhưng **FAR đáng kể (~0,1–0,2)**: mô hình có xu hướng báo động nhầm lưu lượng Normal. Lý do là phân bố train (Attack chiếm đa số) lệch so với test, và một phần Normal ở test có đặc trưng giống tấn công (vd luồng ngắn, TTL bất thường).
- Đường ROC và PR của hai mô hình gần nhau. Có thể giảm FAR bằng cách **dịch ngưỡng quyết định** (vd từ 0,5 lên 0,7) trên đường ROC, đổi lại DR giảm nhẹ, tùy chính sách vận hành SOC.

### 7.2 UNSW-NB15 – Phân loại đa lớp

In [ ]:
evaluate_block('UNSW-NB15', 'multi')

**Nhận xét:**
- Đa lớp trên UNSW-NB15 **khó hơn rõ rệt**: F1-macro thường chỉ khoảng **0,45–0,60** dù F1-weighted cao hơn nhiều. Nguyên nhân là các lớp lớn (Normal, Generic) được dự đoán tốt, còn các lớp nhỏ thì không.
- Confusion matrix chuẩn hóa cho thấy **DoS, Analysis, Backdoor bị nhầm nặng sang Exploits** (và ngược lại). Đây là hiện tượng đã được ghi nhận trong nhiều nghiên cứu: các lớp này có đặc trưng luồng gần như trùng nhau, khác biệt chủ yếu nằm ở payload, mà dataset không cung cấp. Sau khi loại trùng lặp, rất nhiều mẫu DoS/Exploits mâu thuẫn nhãn cũng biến mất, khiến bài toán thực tế hơn nhưng khó hơn.
- Generic và Normal có recall rất cao. Reconnaissance và Fuzzers ở mức trung bình (Fuzzers hay bị nhầm với Normal). **Worms** ít mẫu nên dao động mạnh.
- Với bài toán đa lớp, DR/FAR (quy về Attack/Normal) vẫn tốt, tức mô hình **vẫn phát hiện được có tấn công**, chỉ là gán sai loại tấn công.

### 7.3 CIC-IDS2017 – Phân loại nhị phân

In [ ]:
evaluate_block('CIC-IDS2017', 'binary')

**Nhận xét:**
- CIC-IDS2017 nhị phân thường cho kết quả **gần 100%** (F1 > 0,99, ROC-AUC ≈ 0,999+, FAR < 0,5%), trong đó Random Forest thường nhỉnh hơn CNN-BiLSTM một chút.
- **Kiểm tra rò rỉ (theo yêu cầu khi kết quả gần 100%):** (i) đã loại toàn bộ cột định danh (`Flow ID`, IP, `Timestamp`, cổng nguồn) nếu chúng có trong dữ liệu; (ii) đã loại trùng lặp **trước** khi chia tập, và bảng ở mục 4.5 xác nhận test gần như không trùng train; (iii) scaler và bộ chọn cột chỉ fit trên train. Như vậy kết quả cao **không do rò rỉ kỹ thuật**, mà do bản chất dataset: các cuộc tấn công được sinh bằng công cụ cố định (Hulk, Patator, LOIC…) trong một testbed, tạo ra chữ ký thống kê rất ổn định (`Destination Port`, `Init_Win_bytes`, kích thước gói…).
- Hệ quả: điểm cao trên CIC-IDS2017 **không đảm bảo** mô hình hoạt động tốt trên mạng thật. Đây là hạn chế được nhiều nghiên cứu chỉ ra (Engelen et al., 2021; Arp et al., 2022). Ngoài ra, CIC-IDS2017 gốc còn có lỗi gán nhãn và lỗi trích xuất luồng của CICFlowMeter.

### 7.4 CIC-IDS2017 – Phân loại đa lớp

In [ ]:
evaluate_block('CIC-IDS2017', 'multi')

**Nhận xét:**
- Các lớp lớn (Benign, DoS, DDoS, PortScan, Brute Force) đạt F1 ≈ 0,99+. **Bot** và **Web Attack** thấp hơn (F1 khoảng 0,7–0,95): Bot có lưu lượng giống Benign (kết nối định kỳ tới C&C), còn Web Attack thì Brute Force/XSS trên HTTP rất giống truy cập web bình thường ở mức luồng.
- **Infiltration** (vài mẫu test) và **Heartbleed** (1–2 mẫu test) có metric dao động rất mạnh: chỉ cần đoán sai 1 mẫu là F1 có thể giảm từ 1,0 xuống 0,67 hoặc thấp hơn. Vì vậy F1-macro bị kéo xuống bởi các lớp siêu hiếm này, và cần đọc **F1 từng lớp** thay vì chỉ nhìn một con số.
- Nhờ class_weight, recall của lớp hiếm được cải thiện, đổi lại precision giảm (một số mẫu Benign bị gán thành Bot/Web Attack). Đây là đánh đổi điển hình trong IDS.

### 7.5 Phân tích lỗi – các lớp hay bị nhầm
Với mỗi dataset (bài toán đa lớp), liệt kê 10 cặp **(lớp thật → lớp bị dự đoán)** có số mẫu nhầm lớn nhất, kèm F1 từng lớp và số mẫu train để xem mối liên hệ giữa “ít mẫu” và “sai nhiều”.

In [ ]:
def error_analysis(key, task='multi', k=10):
    d = DATA[key]
    classes = d['classes'][task]
    n = len(classes)
    for m in MODEL_NAMES:
        if (key, task, m) not in CMS:
            continue
        cm = CMS[(key, task, m)]
        rows = [(classes[i], classes[j], int(cm[i, j]), cm[i, j] / max(cm[i].sum(), 1) * 100)
                for i in range(n) for j in range(n) if i != j and cm[i, j] > 0]
        top = pd.DataFrame(rows, columns=['Lớp thật', 'Bị dự đoán thành', 'Số mẫu nhầm', '% của lớp thật'])
        top = top.sort_values('Số mẫu nhầm', ascending=False).head(k).reset_index(drop=True)
        print(f'{m} – {key}: {k} cặp nhầm lẫn lớn nhất')
        display(top.round(2))
    f1_tbl = pd.DataFrame({m: PER_CLASS_F1[(key, task, m)] for m in MODEL_NAMES if (key, task, m) in PER_CLASS_F1})
    f1_tbl.insert(0, 'Số mẫu test', np.bincount(d['y'][task][2], minlength=n))
    f1_tbl.insert(0, 'Số mẫu train', np.bincount(d['y'][task][0], minlength=n))
    print(f'{key}: F1 từng lớp (sắp xếp tăng dần theo F1 của mô hình đầu tiên)')
    display(f1_tbl.sort_values(f1_tbl.columns[2]).round(4))

for key in ACTIVE:
    print('#' * 100)
    error_analysis(key)

**Nhận xét – nguyên nhân lỗi:**

| Dataset | Cặp nhầm điển hình | Nguyên nhân |
|---|---|---|
| UNSW-NB15 | DoS ↔ Exploits, Analysis/Backdoor → Exploits/DoS | Đặc trưng luồng gần như giống nhau (cùng giao thức, cùng TTL, luồng ngắn). Khác biệt nằm ở payload, mà dataset không có. Analysis/Backdoor lại ít mẫu |
| UNSW-NB15 | Fuzzers ↔ Normal, Reconnaissance → Exploits | Fuzzers gửi dữ liệu ngẫu nhiên qua kết nối hợp lệ nên trông giống lưu lượng thường. Recon/Exploits đều là chuỗi luồng ngắn dò dịch vụ |
| CIC-IDS2017 | Bot → Benign, Web Attack → Benign | Bot liên lạc C&C bằng HTTP định kỳ, Web Attack (Brute Force/XSS) là request HTTP hợp lệ về mặt luồng. Cả hai đều ít mẫu |
| CIC-IDS2017 | Infiltration → Benign | Chỉ vài chục mẫu; hành vi sau xâm nhập (quét nội bộ, tải file) giống người dùng bình thường |
| CIC-IDS2017 | DoS ↔ DDoS (rất ít) | Cùng cơ chế flood HTTP; DDoS khác chủ yếu ở số nguồn, mà thông tin này không có trong 1 luồng |

**Kết luận:** lỗi tập trung ở (i) các lớp **ít mẫu** và (ii) các lớp có **đặc trưng luồng chồng lấn**. Muốn cải thiện cần thêm đặc trưng nội dung/payload, đặc trưng ngữ cảnh theo cửa sổ thời gian (nhiều luồng), hoặc thu thập thêm dữ liệu cho lớp hiếm. Chỉ tinh chỉnh mô hình thì khó cải thiện nhiều.

### 7.6 (Tùy chọn) Giải thích mô hình bằng SHAP – Random Forest nhị phân
Dùng `shap.TreeExplainer` trên `SHAP_SAMPLES` mẫu test. Để giảm thời gian (cây RF không giới hạn độ sâu nên tính SHAP rất chậm), ta tính trên **`SHAP_TREES` cây đầu tiên** của rừng. Đây là xấp xỉ hợp lý vì các cây được huấn luyện độc lập, cùng phân phối. Cell được bọc `try/except` để không làm gián đoạn Run All.

In [ ]:
if CONFIG['RUN_SHAP']:
    try:
        import shap
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap'], check=False)
        import shap
    for key in ACTIVE:
        try:
            rf = RF_MODELS[(key, 'binary')]
            rf_small = copy.copy(rf)
            rf_small.estimators_ = rf.estimators_[:CONFIG['SHAP_TREES']]
            rf_small.n_estimators = len(rf_small.estimators_)
            Xt = DATA[key]['X_test']
            idx = np.random.RandomState(SEED).choice(len(Xt), size=min(CONFIG['SHAP_SAMPLES'], len(Xt)), replace=False)
            Xs = pd.DataFrame(Xt[idx], columns=DATA[key]['feature_names'])
            t0 = time.time()
            sv = shap.TreeExplainer(rf_small).shap_values(Xs, check_additivity=False)
            if isinstance(sv, list):
                sv = sv[1]
            elif np.ndim(sv) == 3:
                sv = sv[:, :, 1]
            print(f'{key}: tính SHAP xong trong {time.time() - t0:.1f}s')
            plt.figure()
            shap.summary_plot(sv, Xs, max_display=20, show=False)
            plt.title(f'SHAP summary – Random Forest – {key} – Nhị phân (lớp Attack)')
            save_fig(f'shap_summary_{key}'); plt.show()
        except Exception as e:
            print(f'{key}: bỏ qua SHAP do lỗi -> {e}')
else:
    print('RUN_SHAP = False -> bỏ qua SHAP.')

**Nhận xét:**
- SHAP summary plot cho biết cả **mức độ quan trọng** (trục dọc, sắp theo trung bình |SHAP|) lẫn **chiều tác động** (màu đỏ là giá trị đặc trưng cao; điểm nằm bên phải nghĩa là đẩy dự đoán về Attack).
- **UNSW-NB15**: `sttl` cao (màu đỏ) đẩy mạnh về Attack, khớp với EDA. `ct_state_ttl`, `sbytes`/`smean` nhỏ cũng đẩy về Attack, trong khi `dbytes`/`dmean` lớn (lưu lượng hai chiều nhiều) đẩy về Normal.
- **CIC-IDS2017**: `Destination Port`, `Init_Win_bytes_*`, các đặc trưng kích thước gói backward chiếm ưu thế. Điều này cho thấy mô hình dựa nhiều vào đặc điểm cấu hình của testbed, và cần thận trọng khi triển khai thực tế.
- Thứ hạng SHAP nhìn chung **nhất quán** với Gini importance ở mục 5.3, làm tăng độ tin cậy cho phần giải thích.

## 8. So sánh hai mô hình
### 8.1 Bảng tổng hợp
Bảng gồm mọi tổ hợp Model × Dataset × Task, đánh giá trên cùng tập test. `Precision`/`Recall` là trung bình macro. `FAR` là False Alarm Rate. Thời gian inference tính bằng ms / 1000 mẫu (RF chạy trên CPU, DL chạy trên GPU).

In [ ]:
COLS = ['Model', 'Dataset', 'Task', 'Accuracy', 'Precision_macro', 'Recall_macro', 'F1_macro', 'F1_weighted',
        'ROC_AUC', 'PR_AUC', 'DR', 'FAR', 'FNR', 'Train_time_s', 'Infer_ms_per_1000', 'Size_MB']
SUMMARY = pd.DataFrame(list(METRICS.values()))[COLS].sort_values(['Dataset', 'Task', 'Model']).reset_index(drop=True)
SUMMARY.to_csv(os.path.join(RES_DIR, 'summary_metrics.csv'), index=False)
display(SUMMARY.round(4))

**Nhận xét:** bảng tổng hợp là cơ sở cho các kết luận ở mục 11. Khi đọc bảng, cần ưu tiên **F1-macro, DR và FAR** thay vì Accuracy. Với UNSW-NB15 đa lớp, chênh lệch giữa F1-weighted và F1-macro rất lớn, cho thấy mô hình tốt ở lớp lớn nhưng yếu ở lớp nhỏ.

### 8.2 So sánh các metric (grouped bar)

In [ ]:
metric_cols = ['Accuracy', 'Precision_macro', 'Recall_macro', 'F1_macro', 'ROC_AUC', 'PR_AUC', 'DR', 'FAR']
combos = SUMMARY[['Dataset', 'Task']].drop_duplicates().values.tolist()
fig, axes = plt.subplots(len(combos), 1, figsize=(15, 4.8 * len(combos)), squeeze=False)
for ax, (ds, tk) in zip(axes.ravel(), combos):
    sub = SUMMARY[(SUMMARY['Dataset'] == ds) & (SUMMARY['Task'] == tk)]
    sub = sub.melt(id_vars='Model', value_vars=metric_cols, var_name='Metric', value_name='Giá trị')
    sns.barplot(data=sub, x='Metric', y='Giá trị', hue='Model', hue_order=MODEL_NAMES, ax=ax)
    for cont in ax.containers:
        ax.bar_label(cont, fmt='%.3f', fontsize=7, rotation=90, padding=2)
    ax.set_ylim(0, 1.15)
    ax.set_title(f'So sánh metric trên test – {ds} – {TASK_VN[tk]}')
    ax.set_xlabel('Metric'); ax.set_ylabel('Giá trị')
    ax.legend(loc='lower right')
plt.tight_layout(); save_fig('compare_metrics_grouped_bar'); plt.show()

**Nhận xét:**
- Ở **bài toán nhị phân**, hai mô hình gần như ngang nhau. Random Forest thường nhỉnh hơn một chút về F1 và có FAR thấp hơn trên CIC-IDS2017. Trên UNSW-NB15, CNN-BiLSTM đôi khi có DR cao hơn nhưng FAR cũng cao hơn.
- Ở **bài toán đa lớp**, khoảng cách lớn hơn: Random Forest thường thắng về F1-macro vì xử lý tốt các lớp ít mẫu (cây có thể tạo lá riêng cho vài mẫu), trong khi mạng nơ-ron cần nhiều mẫu hơn để học một lớp.
- ROC-AUC của cả hai đều rất cao (> 0,95). Đây là ví dụ cho thấy ROC-AUC có thể “đẹp” trên dữ liệu mất cân bằng, nên PR-AUC và F1-macro phân biệt mô hình tốt hơn.

### 8.3 So sánh thời gian & kích thước mô hình

In [ ]:
tmp = SUMMARY.copy()
tmp['Bài toán'] = tmp['Dataset'] + ' | ' + tmp['Task'].map(TASK_VN)
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
specs = [('Train_time_s', 'Thời gian train (giây)'),
         ('Infer_ms_per_1000', 'Thời gian dự đoán (ms / 1000 mẫu)'),
         ('Size_MB', 'Kích thước mô hình (MB)')]
for ax, (col, title) in zip(axes, specs):
    sns.barplot(data=tmp, x='Bài toán', y=col, hue='Model', hue_order=MODEL_NAMES, ax=ax)
    for cont in ax.containers:
        ax.bar_label(cont, fmt='%.2f', fontsize=8, padding=2)
    ax.set_yscale('log')
    ax.set_title(title + ' – thang log')
    ax.set_xlabel('Dataset | Bài toán'); ax.set_ylabel(title)
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); save_fig('compare_time_size'); plt.show()

**Nhận xét:**
- **Thời gian train**: RF (CPU, 4 nhân) và CNN-BiLSTM (GPU) cùng ở mức vài phút. Trên CIC-IDS2017, RF có thể nhanh hơn hoặc chậm hơn tùy tham số. DL cần GPU để có thời gian chấp nhận được; nếu chạy trên CPU sẽ chậm hơn hàng chục lần.
- **Thời gian dự đoán**: cả hai đều ở mức mili-giây / 1000 mẫu, đủ cho xử lý realtime. RF không cần GPU nên dễ triển khai trên server thông thường.
- **Kích thước**: CNN-BiLSTM **nhỏ hơn RF hàng chục đến hàng trăm lần** (khoảng 1–2 MB so với hàng chục đến hàng trăm MB), có lợi khi triển khai trên thiết bị biên (router, IoT gateway).

### 8.4 So sánh F1 theo từng lớp tấn công (đa lớp)

In [ ]:
for key in ACTIVE:
    f1_tbl = pd.DataFrame({m: PER_CLASS_F1[(key, 'multi', m)] for m in MODEL_NAMES if (key, 'multi', m) in PER_CLASS_F1})
    models_present = [m for m in MODEL_NAMES if m in f1_tbl.columns]
    fig, axes = plt.subplots(1, 2, figsize=(19, 0.55 * len(f1_tbl) + 3), gridspec_kw={'width_ratios': [1, 1.6]})
    sns.heatmap(f1_tbl[models_present], annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1, ax=axes[0],
                cbar_kws={'label': 'F1-score'})
    axes[0].set_title(f'{key} – F1 từng lớp (heatmap)')
    axes[0].set_xlabel('Mô hình'); axes[0].set_ylabel('Lớp')
    f1_tbl[models_present].plot.barh(ax=axes[1])
    axes[1].set_xlim(0, 1.05)
    axes[1].set_title(f'{key} – F1 từng lớp (grouped bar)')
    axes[1].set_xlabel('F1-score'); axes[1].set_ylabel('Lớp')
    axes[1].legend(title='Mô hình', loc='lower right')
    plt.tight_layout(); save_fig(f'compare_per_class_f1_{key}'); plt.show()
    if len(models_present) == 2:
        f1_tbl['Chênh lệch (RF - DL)'] = f1_tbl['RandomForest'] - f1_tbl['CNN-BiLSTM']
    display(f1_tbl.round(4))
    f1_tbl.to_csv(os.path.join(RES_DIR, f'per_class_f1_{slug(key)}.csv'))

**Nhận xét:**
- Với các lớp lớn, hai mô hình gần như bằng nhau (chênh lệch < 0,01).
- Chênh lệch tập trung ở **lớp nhỏ/khó**. Trên UNSW-NB15 là Analysis, Backdoor, DoS, Worms; trên CIC-IDS2017 là Bot, Web Attack, Infiltration, Heartbleed. Ở các lớp này, RF thường tốt hơn. CNN-BiLSTM đôi khi có recall cao hơn ở một lớp hiếm nhờ class_weight, nhưng precision lại thấp hơn.
- Cột “Chênh lệch” dương nghĩa là RF tốt hơn ở lớp đó. Nhìn bảng này giúp chọn mô hình theo lớp tấn công quan trọng nhất với tổ chức, chứ không chỉ theo F1-macro chung.

### 8.5 Kiểm định McNemar
Kiểm định xem sự khác biệt về tỉ lệ dự đoán đúng giữa RF và CNN-BiLSTM trên **cùng tập test** có ý nghĩa thống kê hay không.
- `n01` là số mẫu RF đúng mà DL sai; `n10` là số mẫu RF sai mà DL đúng.
- Thống kê χ² (có hiệu chỉnh liên tục) = (|n01 − n10| − 1)² / (n01 + n10), 1 bậc tự do.
- Nếu p < 0,05 thì khác biệt có ý nghĩa thống kê.

In [ ]:
def mcnemar_test(y, p1, p2):
    a, b = (p1 == y), (p2 == y)
    n01, n10 = int(np.sum(a & ~b)), int(np.sum(~a & b))
    stat = (abs(n01 - n10) - 1) ** 2 / (n01 + n10) if (n01 + n10) > 0 else 0.0
    return n01, n10, float(stat), float(chi2.sf(stat, df=1))

mc_rows = []
for key in ACTIVE:
    for task in TASKS:
        if (key, task, 'RandomForest') in RESULTS and (key, task, 'CNN-BiLSTM') in RESULTS:
            y = DATA[key]['y'][task][2]
            n01, n10, stat, p = mcnemar_test(y, RESULTS[(key, task, 'RandomForest')]['y_pred'],
                                             RESULTS[(key, task, 'CNN-BiLSTM')]['y_pred'])
            better = 'RandomForest' if n01 > n10 else ('CNN-BiLSTM' if n10 > n01 else 'Ngang nhau')
            mc_rows.append({'Dataset': key, 'Task': task, 'RF đúng / DL sai (n01)': n01, 'RF sai / DL đúng (n10)': n10,
                            'Chi2': round(stat, 3), 'p-value': p, 'Ý nghĩa (p<0.05)': p < 0.05,
                            'Mô hình tốt hơn': better})
MCNEMAR = pd.DataFrame(mc_rows)
MCNEMAR.to_csv(os.path.join(RES_DIR, 'mcnemar.csv'), index=False)
display(MCNEMAR)

**Nhận xét:** do tập test rất lớn (hàng chục đến hàng trăm nghìn mẫu), ngay cả chênh lệch nhỏ cỡ 0,1–0,5% Accuracy cũng thường cho **p-value rất nhỏ**, tức khác biệt *có ý nghĩa thống kê*. Tuy nhiên “có ý nghĩa thống kê” không đồng nghĩa với “có ý nghĩa thực tế”: cần xem thêm độ lớn chênh lệch (F1, FAR) và chi phí triển khai (thời gian, kích thước, khả năng giải thích).

### 8.6 So sánh giữa hai dataset

In [ ]:
pivot = SUMMARY.pivot_table(index=['Task', 'Model'], columns='Dataset', values=['F1_macro', 'DR', 'FAR', 'PR_AUC'])
display(pivot.round(4))

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
for ax, tk in zip(axes, TASKS):
    sub = SUMMARY[SUMMARY['Task'] == tk]
    sns.barplot(data=sub, x='Model', y='F1_macro', hue='Dataset', order=MODEL_NAMES, ax=ax)
    for cont in ax.containers:
        ax.bar_label(cont, fmt='%.3f', fontsize=9, padding=2)
    ax.set_ylim(0, 1.1)
    ax.set_title(f'F1-macro theo dataset – {TASK_VN[tk]}')
    ax.set_xlabel('Mô hình'); ax.set_ylabel('F1-macro (test)')
plt.tight_layout(); save_fig('compare_datasets_f1'); plt.show()

**Nhận xét – dataset nào khó hơn và vì sao:**
- **UNSW-NB15 khó hơn rõ rệt**, nhất là đa lớp (F1-macro thấp hơn CIC-IDS2017 hàng chục điểm %). Lý do:
  1. Train/test được thiết kế với **phân bố khác nhau** (tỉ lệ Attack 68% so với 55%), nên mô hình phải tổng quát hóa thật sự.
  2. Các lớp DoS/Exploits/Analysis/Backdoor có **đặc trưng luồng chồng lấn**.
  3. Có nhiều bản ghi trùng đặc trưng nhưng khác nhãn (nhãn mâu thuẫn).
  4. Số mẫu ít hơn.
- **CIC-IDS2017 dễ hơn**: các tấn công được sinh bằng công cụ cố định trong cùng testbed, nên có chữ ký thống kê rất rõ (cổng, cửa sổ TCP, kích thước gói). Train/val/test lại được chia ngẫu nhiên từ **cùng phân phối**, nên kết quả gần 100%. Điều này **không phản ánh** khả năng phát hiện tấn công mới.

### 8.7 Thảo luận ưu/nhược điểm

| Tiêu chí | Random Forest | 1D-CNN + BiLSTM |
|---|---|---|
| Độ chính xác | Rất cao, ổn định; thường tốt hơn ở lớp ít mẫu | Tương đương ở nhị phân, thường kém hơn một chút ở đa lớp và lớp hiếm |
| Tốc độ train | Nhanh trên CPU, không cần GPU | Cần GPU; nhạy với batch size và learning rate |
| Tốc độ dự đoán | Nhanh (ms / 1000 mẫu) trên CPU | Nhanh trên GPU, chậm hơn trên CPU |
| Kích thước | Lớn (hàng chục đến hàng trăm MB) | Rất nhỏ (~1–2 MB) |
| Giải thích | Tốt: feature importance, SHAP TreeExplainer chính xác | Khó (hộp đen); cần DeepSHAP/Integrated Gradients |
| Tiền xử lý | Không cần chuẩn hóa | Bắt buộc chuẩn hóa và log-transform |
| Triển khai realtime | Dễ: sklearn/ONNX trên server CPU | Dễ trên GPU/edge (TF Lite) vì mô hình nhỏ; mở rộng tốt sang dữ liệu chuỗi (nhiều luồng liên tiếp) |
| Mở rộng | Khó học quan hệ thời gian giữa các luồng | Dễ mở rộng thành mô hình chuỗi luồng / gói tin thô |

## 9. Demo dự đoán
Hàm `predict(sample, dataset_key, task, model_type)`:
1. Nhận **một bản ghi thô** (dict, `pd.Series` hoặc DataFrame) với tên cột như dữ liệu gốc. Bản ghi có thể chứa cả cột thừa (id, nhãn…) vì chúng sẽ tự bị bỏ.
2. Tiền xử lý bằng `NIDSPreprocessor` **đã fit** (impute, signed-log, One-Hot, scaler).
3. Dự đoán bằng RF (`model_type='rf'`) hoặc CNN-BiLSTM (`model_type='dl'`), rồi trả về nhãn, xác suất của nhãn đó và xác suất của mọi lớp.

Có thể truyền `prep=` và `model=` để dùng mô hình **load lại từ file** (xem mục 10).

In [ ]:
def predict(sample, dataset_key, task='binary', model_type='rf', prep=None, model=None):
    d = DATA[dataset_key]
    prep = prep if prep is not None else d['prep']
    if model is None:
        model = RF_MODELS[(dataset_key, task)] if model_type == 'rf' else DL_MODELS[(dataset_key, task)]
    if isinstance(sample, dict):
        df = pd.DataFrame([sample])
    elif isinstance(sample, pd.Series):
        df = sample.to_frame().T
    else:
        df = pd.DataFrame(sample)
    X = prep.transform(df)
    if model_type == 'rf':
        P = model.predict_proba(X)
    else:
        P = to_proba(model.predict(X[..., np.newaxis], verbose=0))
    classes = d['classes'][task]
    out = []
    for row in P:
        i = int(np.argmax(row))
        out.append({'label': classes[i], 'probability': round(float(row[i]), 4),
                    'probs': {c: round(float(v), 4) for c, v in zip(classes, row)}})
    return out[0] if len(out) == 1 else out

In [ ]:
for key in ACTIVE:
    raw = DEMO_RAW[key]
    rows = []
    for i in range(len(raw)):
        rec = raw.iloc[i]
        row = {'Nhãn thật (nhị phân)': BIN_NAMES[int(rec['y_bin'])], 'Nhãn thật (đa lớp)': rec['y_multi']}
        for task in TASKS:
            for mt, short in (('rf', 'RF'), ('dl', 'DL')):
                store = RF_MODELS if mt == 'rf' else DL_MODELS
                if (key, task) not in store:
                    continue
                r = predict(rec, key, task, mt)
                lab, prob = r['label'], r['probability']
                row[f'{short} – {TASK_VN[task]}'] = f'{lab} ({prob:.3f})'
        rows.append(row)
    print('=' * 100)
    print(f'{key}: dự đoán trên {len(raw)} bản ghi thô ngẫu nhiên của test set (nhãn dự đoán và xác suất)')
    display(pd.DataFrame(rows))

# Ví dụ gọi hàm với 1 bản ghi dạng dict
key0 = ACTIVE[0]
sample_dict = DEMO_RAW[key0].drop(columns=LABEL_COLS).iloc[-1].to_dict()
preview = {k: sample_dict[k] for k in list(sample_dict)[:8]}
print(f'Bản ghi thô ({key0}), 8 trường đầu:', preview, '...')
print('RF  – đa lớp:', predict(sample_dict, key0, 'multi', 'rf'))
print('DL  – đa lớp:', predict(sample_dict, key0, 'multi', 'dl'))

**Nhận xét:** đa số bản ghi được hai mô hình dự đoán đúng với xác suất cao (> 0,9). Các trường hợp sai hoặc xác suất thấp thường rơi vào những cặp lớp đã phân tích ở mục 7.5 (vd DoS/Exploits trên UNSW-NB15, Bot/Benign trên CIC-IDS2017). Hàm `predict` nhận dữ liệu thô nên có thể tích hợp trực tiếp vào API: đầu vào là bản ghi luồng do CICFlowMeter/Argus sinh ra, đầu ra là nhãn và xác suất để ra quyết định cảnh báo.

## 10. Lưu kết quả
Lưu vào `/kaggle/working/`:
- `models/`: Random Forest (`.joblib`, nén), CNN-BiLSTM (`.keras`, đã lưu ở mục 6), `NIDSPreprocessor` và `LabelEncoder` (`.joblib`).
- `results/`: bảng tổng hợp metric, F1 từng lớp, McNemar (`.csv`), learning curve (`.json`), cấu hình và log làm sạch.
- `figures/`: mọi biểu đồ (`.png`), được lưu bằng `save_fig()` ngay khi vẽ.

Lưu ý: khi load `preprocessor_*.joblib` ở notebook khác, cần định nghĩa lại class `NIDSPreprocessor` (copy cell ở mục 4.3) trước khi gọi `joblib.load`.

In [ ]:
t0 = time.time()
for key in ACTIVE:
    s = slug(key)
    joblib.dump(DATA[key]['prep'], os.path.join(MODEL_DIR, f'preprocessor_{s}.joblib'))
    joblib.dump(DATA[key]['le'], os.path.join(MODEL_DIR, f'label_encoder_{s}.joblib'))
    for task in TASKS:
        if (key, task) in RF_MODELS:
            joblib.dump(RF_MODELS[(key, task)], os.path.join(MODEL_DIR, f'rf_{s}_{task}.joblib'), compress=3)

all_f1 = pd.concat({f'{k} | {t} | {m}': v for (k, t, m), v in PER_CLASS_F1.items()}, axis=0)
all_f1.rename('F1').to_frame().to_csv(os.path.join(RES_DIR, 'per_class_f1_all.csv'))
with open(os.path.join(RES_DIR, 'dl_histories.json'), 'w', encoding='utf-8') as f:
    json.dump({f'{k} | {t}': h for (k, t), h in HISTORIES.items()}, f, indent=1)
with open(os.path.join(RES_DIR, 'config_and_cleaning_log.json'), 'w', encoding='utf-8') as f:
    json.dump({'CONFIG': CONFIG, 'CLEAN_LOG': CLEAN_LOG,
               'FEATURES': {k: DATA[k]['feature_names'] for k in ACTIVE},
               'CLASSES': {k: DATA[k]['classes'] for k in ACTIVE},
               'RF_PARAMS': {f'{k} | {t}': r['params'] for (k, t, m), r in RESULTS.items() if m == 'RandomForest'}},
              f, indent=2, ensure_ascii=False, default=str)
print(f'Đã lưu xong trong {time.time() - t0:.1f}s')

files = []
for root, _, fnames in os.walk(OUT_DIR):
    for fn in fnames:
        p = os.path.join(root, fn)
        files.append({'File': os.path.relpath(p, OUT_DIR), 'Size (MB)': round(os.path.getsize(p) / 1024 ** 2, 3)})
display(pd.DataFrame(files).sort_values('File').reset_index(drop=True))

In [ ]:
# Kiểm tra: load lại pipeline + mô hình từ file và dự đoán lại 1 bản ghi thô
key0 = ACTIVE[0]
s0 = slug(key0)
prep_loaded = joblib.load(os.path.join(MODEL_DIR, f'preprocessor_{s0}.joblib'))
rf_loaded = joblib.load(os.path.join(MODEL_DIR, f'rf_{s0}_multi.joblib'))
dl_loaded = keras.models.load_model(os.path.join(MODEL_DIR, f'dl_{s0}_multi.keras'))
rec = DEMO_RAW[key0].iloc[0]
r_mem = predict(rec, key0, 'multi', 'rf')
r_file = predict(rec, key0, 'multi', 'rf', prep=prep_loaded, model=rf_loaded)
d_file = predict(rec, key0, 'multi', 'dl', prep=prep_loaded, model=dl_loaded)
print('Nhãn thật          :', rec['y_multi'])
print('RF (trong bộ nhớ)  :', r_mem['label'], r_mem['probability'])
print('RF (load từ file)  :', r_file['label'], r_file['probability'])
print('DL (load từ file)  :', d_file['label'], d_file['probability'])
assert r_mem['label'] == r_file['label'], 'Kết quả RF load từ file không khớp!'
print('OK – pipeline lưu/load hoạt động đúng.')

### 10.3 Đóng gói kết quả để tải về
Gói toàn bộ `figures/`, `results/` và (tùy chọn) `models/` thành một file **`nids_results.zip`** trong `/kaggle/working/`. File zip kèm `README.txt` tóm tắt cấu hình và bảng metric.
- `ZIP_INCLUDE_MODELS = False`: chỉ gói hình và bảng kết quả. File zip khi đó chỉ vài MB, tải nhanh; mô hình RF có thể nặng hàng trăm MB.
- `ZIP_SKIP_BEST_CKPT = True`: bỏ các checkpoint `*_best.keras`, vì trọng số tốt nhất đã được khôi phục vào mô hình cuối `dl_*.keras`.

**Cách tải:**
1. Khi đang chạy interactive, bấm vào link hiện ra bên dưới cell.
2. Hoặc mở panel **Output** bên phải, thư mục `/kaggle/working`.
3. Sau khi **Save Version**, file nằm ở tab **Output** của notebook.

In [ ]:
# ===== 10.3 Đóng gói kết quả thành file zip =====
import zipfile
from datetime import datetime
from IPython.display import FileLink

ZIP_INCLUDE_MODELS = True    # False -> chỉ gói figures + results (nhẹ)
ZIP_SKIP_BEST_CKPT = True    # bỏ checkpoint *_best.keras (trùng với mô hình cuối)
ZIP_PATH = os.path.join(OUT_DIR, 'nids_results.zip')

def build_readme():
    lines = ['HỆ THỐNG PHÁT HIỆN XÂM NHẬP MẠNG (NIDS) – KẾT QUẢ',
             'Thời điểm đóng gói: ' + datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
             'Dataset: ' + ', '.join(ACTIVE),
             'Mô hình: ' + ', '.join(MODEL_NAMES) + ' | Bài toán: ' + ', '.join(TASKS),
             'Cấu hình: ' + json.dumps(CONFIG, ensure_ascii=False),
             '',
             'Cấu trúc thư mục:',
             '  figures/ : biểu đồ EDA, confusion matrix, ROC/PR, learning curve, so sánh (.png)',
             '  results/ : summary_metrics.csv, per_class_f1_*.csv, mcnemar.csv, dl_histories.json, config_and_cleaning_log.json',
             '  models/  : rf_*.joblib (Random Forest), dl_*.keras (CNN-BiLSTM), preprocessor_*.joblib, label_encoder_*.joblib',
             '  (Load preprocessor_*.joblib cần định nghĩa lại class NIDSPreprocessor ở mục 4.3 của notebook)',
             '',
             'BẢNG TỔNG HỢP METRIC TRÊN TEST SET:']
    if 'SUMMARY' in globals():
        lines.append(SUMMARY.round(4).to_string(index=False))
    return chr(10).join(lines) + chr(10)

def make_results_zip(zip_path=ZIP_PATH, include_models=ZIP_INCLUDE_MODELS):
    if os.path.exists(zip_path):
        os.remove(zip_path)
    folders = [FIG_DIR, RES_DIR] + ([MODEL_DIR] if include_models else [])
    n_files, raw_bytes = 0, 0
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for folder in folders:
            for root, _, fnames in os.walk(folder):
                for fn in sorted(fnames):
                    if ZIP_SKIP_BEST_CKPT and fn.endswith('_best.keras'):
                        continue
                    p = os.path.join(root, fn)
                    zf.write(p, arcname=os.path.relpath(p, OUT_DIR))
                    n_files += 1
                    raw_bytes += os.path.getsize(p)
        zf.writestr('README.txt', build_readme())
    return n_files, raw_bytes

t0 = time.time()
n_files, raw_bytes = make_results_zip()
zip_mb = os.path.getsize(ZIP_PATH) / 1024 ** 2
print(f'Đã đóng gói {n_files} file ({raw_bytes / 1024 ** 2:.1f} MB) -> {ZIP_PATH} ({zip_mb:.1f} MB) trong {time.time() - t0:.1f}s')
if zip_mb > 2000:
    print('! File zip lớn (> 2 GB): cân nhắc đặt ZIP_INCLUDE_MODELS = False để tải nhanh hơn.')
with zipfile.ZipFile(ZIP_PATH) as zf:
    print('Một số file trong zip:', zf.namelist()[:10], '...')
os.chdir(OUT_DIR)   # FileLink cần đường dẫn tương đối so với thư mục làm việc
display(FileLink(os.path.basename(ZIP_PATH)))

## 11. Kết luận & Hướng phát triển
Cell dưới đây **tự động tóm tắt số liệu** từ bảng `SUMMARY`, nên các con số luôn khớp với lần chạy hiện tại.

In [ ]:
print('TÓM TẮT KẾT QUẢ TRÊN TEST SET')
print('=' * 100)
for (ds, tk), g in SUMMARY.groupby(['Dataset', 'Task']):
    best = g.loc[g['F1_macro'].idxmax()]
    print(f'{ds} | {TASK_VN[tk]} -> mô hình tốt nhất theo F1-macro: {best.Model}')
    for _, r in g.iterrows():
        print(f'    {r.Model:<13s} F1-macro={r.F1_macro:.4f} | Acc={r.Accuracy:.4f} | DR={r.DR:.4f} | FAR={r.FAR:.4f}'
              f' | ROC-AUC={r.ROC_AUC:.4f} | train={r.Train_time_s:.0f}s | infer={r.Infer_ms_per_1000:.2f} ms/1000 | size={r.Size_MB:.1f} MB')
wins = SUMMARY.loc[SUMMARY.groupby(['Dataset', 'Task'])['F1_macro'].idxmax(), 'Model'].value_counts()
print('=' * 100)
print('Số bài toán mỗi mô hình thắng (theo F1-macro):', wins.to_dict())

### 11.1 Tóm tắt kết quả chính
*(Đối chiếu số liệu cụ thể với bảng tóm tắt tự động ở trên; các nhận định dưới đây phản ánh xu hướng thường thấy với pipeline này.)*
- **CIC-IDS2017**: cả hai mô hình đạt kết quả rất cao ở bài toán nhị phân (F1 > 0,99, FAR < 1%), còn đa lớp có F1-macro khoảng 0,85–0,95, bị kéo xuống bởi Bot, Web Attack, Infiltration, Heartbleed.
- **UNSW-NB15**: nhị phân có F1 khoảng 0,87–0,92 với DR cao nhưng FAR đáng kể. Đa lớp khó nhất (F1-macro khoảng 0,45–0,60) do DoS/Exploits/Analysis/Backdoor chồng lấn đặc trưng.
- **Random Forest** thường thắng hoặc ngang CNN-BiLSTM về F1-macro, nhất là ở lớp hiếm. **CNN-BiLSTM** nhỏ gọn hơn rất nhiều và có DR cạnh tranh.

### 11.2 Mô hình đề xuất
**Random Forest** được đề xuất làm mô hình triển khai chính, vì:
1. F1-macro và FAR tốt nhất hoặc tương đương.
2. Chạy tốt trên CPU, không cần GPU, dự đoán chỉ vài ms / 1000 luồng.
3. **Giải thích được** (feature importance, SHAP), quan trọng để analyst SOC tin và điều tra cảnh báo.
4. Ít nhạy với siêu tham số và tiền xử lý.

CNN-BiLSTM phù hợp khi cần mô hình **rất nhỏ** để chạy trên thiết bị biên, hoặc làm nền tảng mở rộng sang dữ liệu chuỗi (nhiều luồng liên tiếp, gói tin thô). Một hướng thực tế là **ensemble**: RF cho quyết định chính, DL làm bộ phát hiện thứ hai để tăng DR.

### 11.3 Hạn chế
- **Dữ liệu mô phỏng**: cả hai dataset sinh trong testbed, với công cụ tấn công cố định và lưu lượng nền tổng hợp. Mô hình có thể học “dấu vân tay” của môi trường (TTL, cổng, cửa sổ TCP) thay vì hành vi tấn công tổng quát.
- **Mất cân bằng nghiêm trọng**: các lớp Heartbleed, Infiltration, Worms quá ít mẫu nên metric không ổn định. CIC-IDS2017 còn được lấy mẫu 50% (lớp hiếm giữ nguyên), khiến tỉ lệ lớp hơi khác bản gốc.
- **Chưa kiểm thử trên lưu lượng thực**, cũng chưa đánh giá độ trễ end-to-end (trích xuất luồng, tiền xử lý, dự đoán).
- **Cross-dataset generalization**: hai dataset có bộ đặc trưng khác nhau (Argus/Bro so với CICFlowMeter), nên chưa thể train trên dataset này rồi test trên dataset kia. Các nghiên cứu cho thấy hiệu năng giảm mạnh khi làm vậy.
- Chia ngẫu nhiên CIC-IDS2017 khiến train và test cùng phân phối thời gian. Chia theo ngày (vd train thứ 2–4, test thứ 5–6) sẽ thực tế hơn nhưng làm một số lớp chỉ xuất hiện ở test.
- CIC-IDS2017 gốc có lỗi gán nhãn/trích xuất đã được công bố (Engelen et al., 2021). Có thể dùng bản đã sửa lỗi.

### 11.4 Hướng phát triển
1. **Gradient Boosting (XGBoost/LightGBM/CatBoost)**: thường tốt hơn RF trên dữ liệu bảng, nhanh hơn và nhỏ hơn; hỗ trợ GPU.
2. **Autoencoder / Isolation Forest / Deep SVDD** để phát hiện bất thường, chỉ học từ Normal, nhằm phát hiện **zero-day**.
3. **Transformer cho dữ liệu bảng** (TabTransformer, FT-Transformer, TabNet) và mô hình chuỗi trên nhiều luồng liên tiếp hoặc gói tin thô.
4. **Huấn luyện liên tập dữ liệu**: dùng bộ đặc trưng chung (vd NetFlow v9 của NF-UNSW-NB15/NF-CSE-CIC-IDS2018) để train trên một dataset và test trên dataset khác; áp dụng domain adaptation.
5. **Tối ưu ngưỡng** theo chi phí (cost-sensitive), cân bằng DR và FAR theo chính sách SOC; thêm hiệu chỉnh xác suất (calibration).
6. **Triển khai realtime**: CICFlowMeter/Zeek → Kafka → dịch vụ suy luận (FastAPI/ONNX Runtime/TF Serving) → SIEM/dashboard; giám sát concept drift và huấn luyện lại định kỳ.
7. **Explainable AI và phòng thủ đối kháng**: SHAP/LIME cho từng cảnh báo; đánh giá độ bền trước adversarial examples.

## 12. Tài liệu tham khảo
1. N. Moustafa and J. Slay, *UNSW-NB15: a comprehensive data set for network intrusion detection systems (UNSW-NB15 network data set)*, Military Communications and Information Systems Conference (MilCIS), IEEE, 2015.
2. N. Moustafa and J. Slay, *The evaluation of Network Anomaly Detection Systems: Statistical analysis of the UNSW-NB15 data set and the comparison with the KDD99 data set*, Information Security Journal: A Global Perspective, 25(1–3), 2016.
3. I. Sharafaldin, A. H. Lashkari, A. A. Ghorbani, *Toward Generating a New Intrusion Detection Dataset and Intrusion Traffic Characterization*, Proc. 4th ICISSP, 2018 (CIC-IDS2017).
4. G. Engelen, V. Rimmer, W. Joosen, *Troubleshooting an Intrusion Detection Dataset: the CICIDS2017 Case Study*, IEEE Security and Privacy Workshops (SPW), 2021.
5. D. Arp et al., *Dos and Don'ts of Machine Learning in Computer Security*, USENIX Security Symposium, 2022.
6. L. Breiman, *Random Forests*, Machine Learning, 45(1), 2001.
7. S. Hochreiter, J. Schmidhuber, *Long Short-Term Memory*, Neural Computation, 9(8), 1997.
8. N. V. Chawla et al., *SMOTE: Synthetic Minority Over-sampling Technique*, JAIR, 16, 2002.
9. S. M. Lundberg, S.-I. Lee, *A Unified Approach to Interpreting Model Predictions*, NeurIPS, 2017 (SHAP).
10. Q. McNemar, *Note on the sampling error of the difference between correlated proportions or percentages*, Psychometrika, 12(2), 1947.
11. F. Pedregosa et al., *Scikit-learn: Machine Learning in Python*, JMLR, 12, 2011.
12. M. Abadi et al., *TensorFlow: A System for Large-Scale Machine Learning*, OSDI, 2016; F. Chollet et al., *Keras*, https://keras.io.
13. Thư viện: pandas, NumPy, Matplotlib, seaborn, SciPy, joblib (tài liệu chính thức của từng thư viện).